
# ED Pathway Trainer – PoC (v3, portable) 🚑
**Included:**
- ETA non-negativity (Softplus head) + clamp at inference
- Overfitting mitigations: label smoothing, dropout, 5% label noise
- ORBIS simulator (HL7 ADT/ORM/ORU)
- Order bridge (model actions → ORM^O01)
- Replay loop (process inbox/pending, pump results to notifier)


In [98]:

from pathlib import Path
import json, textwrap, os

def pick_base_dir():
    candidates = [Path("./ed_demo_data"), Path("."), Path("/tmp"), Path("/mnt/data")]
    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            t = p / "__wtest__"; t.write_text("ok"); t.unlink()
            return p
        except Exception:
            continue
    raise RuntimeError("No writable base directory found")

BASE = pick_base_dir()
print("Using BASE:", BASE.resolve())

# FSM (with specialist stage)
BASE_FSM_FALLBACK = {
  "version": "0.2",
  "pathways": {
    "major_trauma":{"states":["ems_prealert","resus_arrival","primary_survey","fast_ct_parallel","results_review","disposition"],"initial":"ems_prealert","events":{}},
    "chest_pain":{"states":["triage","ecg_and_labs","risk_stratify","imaging_optional","observation_or_admit","disposition"],"initial":"triage","events":{}},
    "pregnant_syncope":{"states":["triage","initial_workup","fetal_assessment","routing_decision","disposition"],"initial":"triage","events":{}}
  },
  "shared_triggers": {},
  "tools_required": []
}
SPECIALIST_STAGE = {
  "states": ["init_consult","await_response","negotiation","handover_or_return"],
  "initial": "init_consult",
  "events": {
    "send_consult": {"from":["init_consult"],"to":"await_response","actions":["send_written_consult","call_specialist"]},
    "fast_track_accept": {"from":["init_consult","await_response"],"to":"handover_or_return","actions":["direct_transfer_to_service"]},
    "decline_or_delay": {"from":["await_response"],"to":"negotiation","actions":["initiate_attending_discussion","offer_additional_tests"]},
    "final_decision": {"from":["negotiation"],"to":"handover_or_return","actions":["transfer_if_accept","return_to_ED_if_decline"]}
  }
}
FSM_WITH = BASE / "ai_coordinator_state_machine_with_specialist.json"
if not FSM_WITH.exists():
    data = BASE_FSM_FALLBACK
    data["shared_specialist_coordination"] = SPECIALIST_STAGE
    FSM_WITH.write_text(json.dumps(data, indent=2))
print("FSM:", FSM_WITH.as_posix())

# Reminder engine
REM_PATH = BASE / "reminder_engine.py"
if not REM_PATH.exists():
    REM_PATH.write_text(textwrap.dedent('''
    from datetime import datetime, timedelta
    from typing import Dict, Any, List, Optional, Callable
    class ReminderEngine:
        def __init__(self, now_fn: Callable[[], datetime] = None):
            self.now_fn = now_fn or (lambda: datetime.utcnow())
            self.patients: Dict[str, Dict[str, Any]] = {}
            self.callbacks: List[Callable[[str, str, Dict[str, Any]], None]] = []
        def on_notify(self, fn): self.callbacks.append(fn)
        def _emit(self, pid, key, payload):
            for cb in self.callbacks: cb(pid, key, payload)
        def ensure_patient(self, pid): self.patients.setdefault(pid, {"events": {}, "timers": []})
        def ingest_event(self, pid, event_type, payload=None):
            payload = payload or {}; self.ensure_patient(pid)
            now = self.now_fn(); self.patients[pid]["events"].setdefault(event_type, []).append({"t": now, **payload}); self._evaluate(pid)
        def add_timer(self, pid, key, due_in, condition):
            self.ensure_patient(pid); due_at = self.now_fn() + due_in
            self.patients[pid]["timers"].append({"key":key,"due_at":due_at,"condition":condition,"fired":False})
        def _has_event(self, pid, event_type, where=None):
            evs = self.patients.get(pid,{}).get("events",{}).get(event_type,[])
            if where is None: return bool(evs)
            return any(all(e.get(k)==v for k,v in where.items()) for e in evs)
        def _evaluate(self, pid):
            now = self.now_fn()
            for t in self.patients[pid]["timers"]:
                if t["fired"]: continue
                if now >= t["due_at"]:
                    cond = t["condition"]; ok = True
                    if "require_event" in cond: ok = self._has_event(pid, cond["require_event"], cond.get("where"))
                    if not ok: t["fired"] = True; self._emit(pid, t["key"], {"due_at": t["due_at"], "condition": cond})
        def apply_lab_protocol(self, pid, protocol):
            self.ensure_patient(pid)
            self.add_timer(pid, f"{protocol['test_name']}_initial_order_due", timedelta(minutes=protocol.get("initial_order_due_min",10)),
                           {"require_event":"order_placed","where":{"test":protocol["test_name"]}})
            self.patients[pid].setdefault("lab_protocols", {})[protocol["test_name"]] = protocol
        def on_sample_collected(self, pid, test_name):
            proto = self.patients.get(pid,{}).get("lab_protocols",{}).get(test_name)
            if not proto: return
            for mins in proto.get("repeat_schedule_min",[]):
                self.add_timer(pid, f"{test_name}_repeat_due_{mins}m", timedelta(minutes=mins),
                               {"require_event":"sample_collected","where":{"test":test_name,"offset_min":mins}})
    '''))

# Results notifier (newline-safe)
RES_PATH = BASE / "results_notifier.py"
RES_PATH.write_text(textwrap.dedent('''
from typing import Callable, Dict, Any, List, Tuple
from datetime import datetime
class ResultsNotifier:
    def __init__(self):
        self.callbacks: List[Callable[[str, str, Dict[str, Any]], None]] = []
        self.last_values: Dict[str, Dict[str, Tuple[float, datetime]]] = {}
        self.delta_rules = {"TROPONIN": {"abs_ng_per_l": 5.0, "rel_pct": 20.0}}
    def on_notify(self, fn): self.callbacks.append(fn)
    def _emit(self, pid, event, payload):
        for cb in self.callbacks: cb(pid, event, payload)
    @staticmethod
    def _split_segments(msg: str):
        norm = msg.replace('\r\n','\n').replace('\r','\n')
        return [s.split('|') for s in norm.strip().split('\n') if s]
    @staticmethod
    def _field(component: str, idx: int) -> str:
        parts = component.split('^'); return parts[idx] if idx < len(parts) else ''
    @staticmethod
    def _parse_ts(ts: str):
        for fmt in ('%Y%m%d%H%M%S','%Y%m%d%H%M','%Y%m%d'):
            try: return datetime.strptime(ts, fmt)
            except Exception: pass
        return None
    def _get_pid(self, segs):
        for s in segs:
            if s[0]=='PID': return s[3].split('^')[0] if len(s)>3 else ''
        return ''
    def handle_hl7(self, message: str):
        segs = self._split_segments(message)
        if not segs or segs[0][0] != 'MSH': return
        msg_type = segs[0][8] if len(segs[0])>8 else ''
        pid = self._get_pid(segs) or 'UNKNOWN'
        if 'ORU^R01' in msg_type: self._handle_oru(pid, segs)
        elif 'MDM^T02' in msg_type or ('ORU^R01' in msg_type and any(s[0]=='OBX' and len(s)>2 and s[2] in ('TX','FT','ED') for s in segs)):
            self._handle_report(pid, segs)
    def _handle_oru(self, pid, segs):
        obr_accession, obr_ts = None, None
        for s in segs:
            if s[0]=='OBR':
                obr_accession = s[3] if len(s)>3 else None
                obr_ts = self._parse_ts(s[7]) if len(s)>7 else None
            if s[0]=='OBX':
                id_comp = s[3] if len(s)>3 else ''
                code = self._field(id_comp,0) or self._field(id_comp,1) or 'UNKNOWN_TEST'
                value_raw = s[5] if len(s)>5 else ''
                units = s[6] if len(s)>6 else ''
                status = s[11] if len(s)>11 else ''
                ts = self._parse_ts(s[14]) if len(s)>14 else obr_ts
                try: value = float(value_raw)
                except Exception: value = None
                payload = {'test_code':code.upper(),'value_raw':value_raw,'value':value,'units':units,'status':status,'ts':ts,'accession':obr_accession}
                self._emit(pid,'lab_result_ready',payload)
                if status and status.upper().startswith('C'): self._emit(pid,'lab_result_critical',payload)
                if value is not None: self._maybe_emit_delta(pid, payload)
    def _maybe_emit_delta(self, pid, payload):
        code, value = payload['test_code'], payload['value']
        ts = payload['ts'] or datetime.utcnow()
        rules = self.delta_rules.get(code)
        if not rules:
            self.last_values.setdefault(pid,{})[code] = (value, ts); return
        last = self.last_values.get(pid,{}).get(code)
        if last:
            prev, _ = last
            abs_delta = abs(value - prev)
            rel_pct = (abs_delta/prev*100.0) if prev else 0.0
            if (abs_delta >= rules.get('abs_ng_per_l', 1e9)) or (rel_pct >= rules.get('rel_pct', 1e9)):
                delta_payload = {**payload,'prev_value':prev,'abs_delta':abs_delta,'rel_pct':rel_pct}
                self._emit(pid,'lab_delta_positive',delta_payload)
        self.last_values.setdefault(pid,{})[code] = (value, ts)
    def _handle_report(self, pid, segs):
        text_blocks, study_id, ts = [], None, None
        for s in segs:
            if s[0]=='OBR':
                study_id = s[3] if len(s)>3 else study_id
                ts = self._parse_ts(s[7]) if len(s)>7 else ts
            if s[0]=='OBX' and len(s)>2 and s[2] in ('TX','FT','ED'):
                text_blocks.append(s[5] if len(s)>5 else '')
        if text_blocks:
            self._emit(pid,'imaging_report_ready',{'study_id':study_id,'report_text':'\n'.join(text_blocks),'ts':ts})
'''))
print("Helpers ready:", (BASE/'reminder_engine.py').exists(), (BASE/'results_notifier.py').exists())

# ORBIS simulator
SIM_PATH = BASE / "orbis_sim.py"
if not SIM_PATH.exists():
    SIM_PATH.write_text(textwrap.dedent('''
    from pathlib import Path
    import json, time, random
    from datetime import datetime, timedelta
    BASE = Path("./ed_demo_data")
    SIM_DIR = BASE / "orbis_sim"
    INBOX = SIM_DIR / "inbox"; OUTBOX = SIM_DIR / "outbox"; PROCESSED = SIM_DIR / "processed"; STATE = SIM_DIR / "state.json"
    for p in (INBOX, OUTBOX, PROCESSED): p.mkdir(parents=True, exist_ok=True)
    def _ts(dt=None): dt = dt or datetime.utcnow(); return dt.strftime("%Y%m%d%H%M%S")
    def _join(segs): return "\r".join(segs) + "\r"
    def _msh(msg_type, msg_id=None, sending_app="ORBIS", sending_fac="HOSP", recv_app="ED", recv_fac="HOSP"):
        msg_id = msg_id or f"MSG{int(time.time()*1000)}"; return f"MSH|^~\&|{sending_app}|{sending_fac}|{recv_app}|{recv_fac}|{_ts()}||{msg_type}|{msg_id}|P|2.5"
    def _pid(patient_id="P-DEM", last="DOE", first="JANE", dob="19800101", sex="F"): return f"PID|||{patient_id}||{last}^{first}||{dob}|{sex}"
    def hl7_adt_a04(patient_id, **kw): return _join([_msh("ADT^A04"), _pid(patient_id, **kw), f"PV1||E||||||||||||||||||"])
    def hl7_orm_o01(patient_id, placer="PLACER1", filler="FILLER1", code="TROPONIN^Troponin I", when=None):
        when = when or _ts(); return _join([_msh("ORM^O01"), _pid(patient_id), f"ORC|NW|{placer}|{filler}|||||{when}", f"OBR|1|{placer}|{filler}|{code}|||{when}"])
    def hl7_oru_r01_numeric(patient_id, accession, code="TROPONIN^Troponin I", value="12.0", units="ng/L", status="F", obs_time=None):
        obs_time = obs_time or _ts(); return _join([_msh("ORU^R01"), _pid(patient_id), f"OBR|1|{accession}|{accession}|{code}|||{obs_time}", f"OBX|1|NM|{code}||{value}|{units}|||N||{status}|||{obs_time}"])
    def hl7_oru_r01_text(patient_id, accession, code="CXR^Chest X-ray", text="Final report available.", status="F", obs_time=None):
        obs_time = obs_time or _ts(); return _join([_msh("ORU^R01"), _pid(patient_id), f"OBR|1|{accession}|{accession}|{code}|||{obs_time}", f"OBX|1|TX|{code}||{text}||||||{status}|||{obs_time}"])
    def _load_state():
        if STATE.exists():
            try: return json.loads(STATE.read_text())
            except Exception: pass
        return {"pending": []}
    def _save_state(st): STATE.write_text(json.dumps(st, indent=2))
    def admit_patient(patient_id="P-DEMO"):
        out = OUTBOX / f"{_ts()}_ADT_A04_{patient_id}.hl7"; out.write_text(hl7_adt_a04(patient_id)); return out.as_posix()
    def place_order(patient_id, code, delay_sec=60):
        placer = f"PLC{int(time.time())}"; filler = f"ACC{int(time.time())}"; inpath = INBOX / f"{_ts()}_ORM_O01_{patient_id}_{placer}.hl7"
        inpath.write_text(hl7_orm_o01(patient_id, placer=placer, filler=filler, code=code))
        st = _load_state(); due_at = (datetime.utcnow() + timedelta(seconds=delay_sec)).timestamp()
        st["pending"].append({"patient_id": patient_id, "accession": filler, "code": code, "due_at": due_at}); _save_state(st); return inpath.as_posix()
    def _emit_result(item):
        pid, acc, code = item["patient_id"], item["accession"], item["code"]
        if code.startswith("TROPONIN"):
            import random; value = f"{round(random.uniform(5.0, 60.0), 1)}"; msg = hl7_oru_r01_numeric(pid, acc, code=code, value=value, units="ng/L")
        elif code.startswith("CT_"):
            msg = hl7_oru_r01_text(pid, acc, code=code, text="CT complete. No acute bleed.")
        elif code.startswith("CXR"):
            msg = hl7_oru_r01_text(pid, acc, code=code, text="No acute cardiopulmonary disease.")
        else:
            msg = hl7_oru_r01_text(pid, acc, code=code, text="Report ready.")
        out = OUTBOX / f"{_ts()}_ORU_R01_{pid}_{acc}.hl7"; out.write_text(msg); return out.as_posix()
    def process_inbox_and_pending(max_to_process=50):
        st = _load_state(); now = time.time(); emitted=[]; still=[]
        for it in st["pending"]:
            if it["due_at"] <= now: emitted.append(_emit_result(it))
            else: still.append(it)
        st["pending"] = still; _save_state(st)
        moved=[]; 
        for f in sorted(INBOX.glob("*.hl7"))[:max_to_process]:
            (PROCESSED/f.name).write_text(f.read_text()); f.unlink(); moved.append((PROCESSED/f.name).as_posix())
        return {"emitted_oru": emitted, "moved_inbox": moved}
    def pump_to_notifier(results_notifier, delete_after=True, limit=100):
        processed=[]; 
        for f in sorted(OUTBOX.glob("*.hl7"))[:limit]:
            msg = f.read_text()
            try:
                results_notifier.handle_hl7(msg); processed.append(f.as_posix())
                if delete_after: (PROCESSED/f.name).write_text(msg); f.unlink()
            except Exception as e: print("Notifier error:", f.name, e)
        return processed
    '''))
print("ORBIS sim:", SIM_PATH.as_posix())


Using BASE: /kaggle/working/ed_demo_data
FSM: ed_demo_data/ai_coordinator_state_machine_with_specialist.json
Helpers ready: True True
ORBIS sim: ed_demo_data/orbis_sim.py


In [99]:
# Cell 2 — helpers (INLINE: ResultsNotifier + ORBIS simulator). No file imports that can break on "\r".

from importlib.util import spec_from_file_location, module_from_spec
from pathlib import Path
from datetime import datetime, timedelta
from typing import Callable, Dict, Any, List, Tuple
import json, time, random
import torch, torch.nn as nn

def _import_by_path(name, path):
    spec = spec_from_file_location(name, path)
    mod = module_from_spec(spec); spec.loader.exec_module(mod); return mod

# --- ReminderEngine stays file-based (written in Cell 1) ---
ReminderEngine = _import_by_path("reminder_engine", BASE / "reminder_engine.py").ReminderEngine

# --- INLINE ResultsNotifier (newline-safe) ---
class ResultsNotifier:
    def __init__(self):
        self.callbacks: List[Callable[[str, str, Dict[str, Any]], None]] = []
        self.last_values: Dict[str, Dict[str, Tuple[float, datetime]]] = {}
        self.delta_rules = {"TROPONIN": {"abs_ng_per_l": 5.0, "rel_pct": 20.0}}
    def on_notify(self, fn): self.callbacks.append(fn)
    def _emit(self, pid, event, payload): 
        for cb in self.callbacks: cb(pid, event, payload)
    @staticmethod
    def _split_segments(msg: str):
        norm = msg.replace('\r\n','\n').replace('\r','\n')
        return [s.split('|') for s in norm.strip().split('\n') if s]
    @staticmethod
    def _field(component: str, idx: int) -> str:
        parts = component.split('^'); return parts[idx] if idx < len(parts) else ''
    @staticmethod
    def _parse_ts(ts: str):
        for fmt in ('%Y%m%d%H%M%S','%Y%m%d%H%M','%Y%m%d'):
            try: return datetime.strptime(ts, fmt)
            except Exception: pass
        return None
    def _get_pid(self, segs):
        for s in segs:
            if s and s[0]=='PID':
                return (s[3].split('^')[0] if len(s)>3 and s[3] else '') or ''
        return ''
    def handle_hl7(self, message: str):
        segs = self._split_segments(message)
        if not segs or segs[0][0] != 'MSH': return
        msg_type = segs[0][8] if len(segs[0])>8 else ''
        pid = self._get_pid(segs) or 'UNKNOWN'
        if 'ORU^R01' in msg_type: self._handle_oru(pid, segs)
        elif 'MDM^T02' in msg_type or ('ORU^R01' in msg_type and any(s[0]=='OBX' and len(s)>2 and s[2] in ('TX','FT','ED') for s in segs)):
            self._handle_report(pid, segs)
    def _handle_oru(self, pid, segs):
        obr_accession, obr_ts = None, None
        for s in segs:
            if not s: continue
            if s[0]=='OBR':
                obr_accession = s[3] if len(s)>3 else None
                obr_ts = self._parse_ts(s[7]) if len(s)>7 else None
            if s[0]=='OBX':
                id_comp = s[3] if len(s)>3 else ''
                code = self._field(id_comp,0) or self._field(id_comp,1) or 'UNKNOWN_TEST'
                value_raw = s[5] if len(s)>5 else ''
                units = s[6] if len(s)>6 else ''
                status = s[11] if len(s)>11 else ''
                ts = self._parse_ts(s[14]) if len(s)>14 else obr_ts
                try: value = float(value_raw)
                except Exception: value = None
                payload = {'test_code':code.upper(),'value_raw':value_raw,'value':value,'units':units,'status':status,'ts':ts,'accession':obr_accession}
                self._emit(pid,'lab_result_ready',payload)
                if status and status.upper().startswith('C'): self._emit(pid,'lab_result_critical',payload)
                if value is not None: self._maybe_emit_delta(pid, payload)
    def _maybe_emit_delta(self, pid, payload):
        code, value = payload['test_code'], payload['value']
        ts = payload['ts'] or datetime.utcnow()
        rules = self.delta_rules.get(code)
        if not rules:
            self.last_values.setdefault(pid,{})[code] = (value, ts); return
        last = self.last_values.get(pid,{}).get(code)
        if last:
            prev, _ = last
            abs_delta = abs(value - prev)
            rel_pct = (abs_delta/prev*100.0) if prev else 0.0
            if (abs_delta >= rules.get('abs_ng_per_l', 1e9)) or (rel_pct >= rules.get('rel_pct', 1e9)):
                self._emit(pid,'lab_delta_positive',{**payload,'prev_value':prev,'abs_delta':abs_delta,'rel_pct':rel_pct})
        self.last_values.setdefault(pid,{})[code] = (value, ts)
    def _handle_report(self, pid, segs):
        text_blocks, study_id, ts = [], None, None
        for s in segs:
            if not s: continue
            if s[0]=='OBR':
                study_id = s[3] if len(s)>3 else study_id
                ts = self._parse_ts(s[7]) if len(s)>7 else ts
            if s[0]=='OBX' and len(s)>2 and s[2] in ('TX','FT','ED'):
                text_blocks.append(s[5] if len(s)>5 else '')
        if text_blocks:
            self._emit(pid,'imaging_report_ready',{'study_id':study_id,'report_text':'\n'.join(text_blocks),'ts':ts})

# --- INLINE ORBIS simulator (no import from disk; safe '\r' handling) ---
class OrbisSim:
    def __init__(self, base: Path):
        self.BASE = Path(base)
        self.SIM_DIR = self.BASE / "orbis_sim"
        self.INBOX = self.SIM_DIR / "inbox"
        self.OUTBOX = self.SIM_DIR / "outbox"
        self.PROCESSED = self.SIM_DIR / "processed"
        self.STATE = self.SIM_DIR / "state.json"
        for p in (self.INBOX, self.OUTBOX, self.PROCESSED):
            p.mkdir(parents=True, exist_ok=True)
        if not self.STATE.exists():
            self._save_state({"pending": []})
    @staticmethod
    def _ts(dt=None):
        dt = dt or datetime.utcnow()
        return dt.strftime("%Y%m%d%H%M%S")
    @staticmethod
    def _join(segs):
        return '\r'.join(segs) + '\r'  # HL7 requires CR delimiters
    def _msh(self, msg_type, msg_id=None, sending_app="ORBIS", sending_fac="HOSP", recv_app="ED", recv_fac="HOSP"):
        msg_id = msg_id or f"MSG{int(time.time()*1000)}"
        return f"MSH|^~\\&|{sending_app}|{sending_fac}|{recv_app}|{recv_fac}|{self._ts()}||{msg_type}|{msg_id}|P|2.5"
    @staticmethod
    def _pid(patient_id="P-DEM", last="DOE", first="JANE", dob="19800101", sex="F"):
        return f"PID|||{patient_id}||{last}^{first}||{dob}|{sex}"
    def _load_state(self):
        try: return json.loads(self.STATE.read_text())
        except Exception: return {"pending": []}
    def _save_state(self, st): self.STATE.write_text(json.dumps(st, indent=2))
    # Message builders
    def hl7_adt_a04(self, patient_id, **kw):
        return self._join([self._msh("ADT^A04"), self._pid(patient_id, **kw), "PV1||E||||||||||||||||||"])
    def hl7_orm_o01(self, patient_id, placer="PLACER1", filler="FILLER1", code="TROPONIN^Troponin I", when=None):
        when = when or self._ts()
        return self._join([self._msh("ORM^O01"), self._pid(patient_id),
                           f"ORC|NW|{placer}|{filler}|||||{when}",
                           f"OBR|1|{placer}|{filler}|{code}|||{when}"])
    def hl7_oru_r01_numeric(self, patient_id, accession, code="TROPONIN^Troponin I", value="12.0", units="ng/L", status="F", obs_time=None):
        obs_time = obs_time or self._ts()
        return self._join([self._msh("ORU^R01"), self._pid(patient_id),
                           f"OBR|1|{accession}|{accession}|{code}|||{obs_time}",
                           f"OBX|1|NM|{code}||{value}|{units}|||N||{status}|||{obs_time}"])
    def hl7_oru_r01_text(self, patient_id, accession, code="CXR^Chest X-ray", text="Report ready.", status="F", obs_time=None):
        obs_time = obs_time or self._ts()
        return self._join([self._msh("ORU^R01"), self._pid(patient_id),
                           f"OBR|1|{accession}|{accession}|{code}|||{obs_time}",
                           f"OBX|1|TX|{code}||{text}||||||{status}|||{obs_time}"])
    # Public API
    def admit_patient(self, patient_id="P-DEMO"):
        out = self.OUTBOX / f"{self._ts()}_ADT_A04_{patient_id}.hl7"
        out.write_text(self.hl7_adt_a04(patient_id))
        return out.as_posix()
    def place_order(self, patient_id, code, delay_sec=60):
        placer = f"PLC{int(time.time())}"; filler = f"ACC{int(time.time())}"
        inpath = self.INBOX / f"{self._ts()}_ORM_O01_{patient_id}_{placer}.hl7"
        inpath.write_text(self.hl7_orm_o01(patient_id, placer=placer, filler=filler, code=code))
        st = self._load_state()
        due_at = (datetime.utcnow() + timedelta(seconds=delay_sec)).timestamp()
        st["pending"].append({"patient_id": patient_id, "accession": filler, "code": code, "due_at": due_at})
        self._save_state(st)
        return inpath.as_posix()
    def _emit_result(self, item):
        pid, acc, code = item["patient_id"], item["accession"], item["code"]
        if code.startswith("TROPONIN"):
            value = f"{round(random.uniform(5.0, 60.0), 1)}"
            msg = self.hl7_oru_r01_numeric(pid, acc, code=code, value=value, units="ng/L")
        elif code.startswith("CT_"):
            msg = self.hl7_oru_r01_text(pid, acc, code=code, text="CT complete. No acute bleed.")
        elif code.startswith("CXR"):
            msg = self.hl7_oru_r01_text(pid, acc, code=code, text="No acute cardiopulmonary disease.")
        else:
            msg = self.hl7_oru_r01_text(pid, acc, code=code, text="Final report available.")
        out = self.OUTBOX / f"{self._ts()}_ORU_R01_{pid}_{acc}.hl7"
        out.write_text(msg)
        return out.as_posix()
    def process_inbox_and_pending(self, max_to_process=50):
        st = self._load_state(); now = time.time()
        emitted, still = [], []
        for it in st["pending"]:
            if it["due_at"] <= now: emitted.append(self._emit_result(it))
            else: still.append(it)
        st["pending"] = still; self._save_state(st)
        moved = []
        for f in sorted(self.INBOX.glob("*.hl7"))[:max_to_process]:
            (self.PROCESSED/f.name).write_text(f.read_text()); f.unlink(); moved.append((self.PROCESSED/f.name).as_posix())
        return {"emitted_oru": emitted, "moved_inbox": moved}
    def pump_to_notifier(self, results_notifier, delete_after=True, limit=100):
        processed = []
        for f in sorted(self.OUTBOX.glob("*.hl7"))[:limit]:
            msg = f.read_text()
            try:
                results_notifier.handle_hl7(msg); processed.append(f.as_posix())
                if delete_after: (self.PROCESSED/f.name).write_text(msg); f.unlink()
            except Exception as e:
                print("Notifier error:", f.name, e)
        return processed

# Instantiate helpers
rn = ResultsNotifier()
rn.on_notify(lambda pid, ev, payload: print(f"[NOTIFY] {pid} - {ev} - {(payload.get('test_code') or payload.get('study_id') or '')}"))
re = ReminderEngine()
re.on_notify(lambda pid, key, payload: print(f"[REMINDER] {pid} - {key} - due @ {payload['due_at']}"))
orbis_sim = OrbisSim(BASE)

print("Helpers loaded: inline ResultsNotifier + ORBIS simulator; file-based ReminderEngine.")


Helpers loaded: inline ResultsNotifier + ORBIS simulator; file-based ReminderEngine.


In [100]:
# After Cell 2, monkey-patch the inline class
def _handle_oru_numeric_only(self, pid, segs):
    obr_accession, obr_ts = None, None
    for s in segs:
        if not s: continue
        if s[0]=='OBR':
            obr_accession = s[3] if len(s)>3 else None
            obr_ts = self._parse_ts(s[7]) if len(s)>7 else None
        if s[0]=='OBX':
            if len(s)>2 and s[2] in ('TX','FT','ED'):  # skip text here
                continue
            id_comp = s[3] if len(s)>3 else ''
            code = self._field(id_comp,0) or self._field(id_comp,1) or 'UNKNOWN_TEST'
            value_raw = s[5] if len(s)>5 else ''; units = s[6] if len(s)>6 else ''
            status = s[11] if len(s)>11 else ''; ts = self._parse_ts(s[14]) if len(s)>14 else obr_ts
            try: value = float(value_raw)
            except Exception: value = None
            payload = {'test_code':code.upper(),'value_raw':value_raw,'value':value,'units':units,'status':status,'ts':ts,'accession':obr_accession}
            self._emit(pid,'lab_result_ready',payload)
            if status and status.upper().startswith('C'): self._emit(pid,'lab_result_critical',payload)
            if value is not None: self._maybe_emit_delta(pid, payload)

ResultsNotifier._handle_oru = _handle_oru_numeric_only


In [101]:

random.seed(42); np.random.seed(42)
ACTIONS = ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"]
ACTION2ID = {a:i for i,a in enumerate(ACTIONS)}

from dataclasses import dataclass
@dataclass
class CaseConfig:
    duration_min: int = 180
    bin_size_min: int = 5
    p_fast_track: float = 0.15
    p_neuro_turf: float = 0.25
    base_load: int = 50
    label_noise: float = 0.05

def generate_case(cfg: CaseConfig, pid: int):
    steps = cfg.duration_min // cfg.bin_size_min
    seq = []
    is_trauma = np.random.rand() < 0.3
    triage_cat = np.random.choice([1,2,3,4], p=[0.1,0.35,0.4,0.15])
    complaint = np.random.choice([0,1], p=[0.6,0.4])
    cap_icu = np.random.choice([0,1], p=[0.7,0.3])
    fast_track = (np.random.rand() < cfg.p_fast_track) and (complaint==1 or np.random.rand()<0.1)
    neuro_turf = (np.random.rand() < cfg.p_neuro_turf) and (complaint==0)
    t_first_lab = np.random.randint(2, 10)
    t_first_imaging = np.random.randint(6, 18)
    decision_ready_t = np.random.randint(18, steps-1)
    for t in range(steps):
        vit_hr = int(np.clip(np.random.normal(85,12), 45, 160))
        vit_map = int(np.clip(np.random.normal(85,10), 50, 120))
        capacity_stale = 1 if np.random.rand() < 0.1 else 0
        action = "none"
        if t == 1: action = "place_labs"
        if t == t_first_imaging: action = "place_xray" if not is_trauma else "place_ct"
        if fast_track and 2 <= t < 6: action = "page_cardiology" if complaint==0 else "prepare_admit_icu"
        if neuro_turf and 6 <= t < 12: action = "page_neurology"
        if t == decision_ready_t: action = "prepare_admit_icu" if (triage_cat<=2 and cap_icu==1) else "prepare_admit_ward"
        if np.random.rand() < cfg.label_noise:
            action = np.random.choice(["place_labs","place_xray","place_ct","page_neurology","none"])
        eta_first_result = max(0, (t_first_lab - t) * cfg.bin_size_min)
        eta_decision = max(0, (decision_ready_t - t) * cfg.bin_size_min)
        seq.append({"pid":pid,"t":t,"triage":triage_cat,"complaint":complaint,"hr":vit_hr,"map":vit_map,"cap_icu":cap_icu,
                    "capacity_stale":capacity_stale,"action":ACTION2ID[action],"eta_first_result":eta_first_result,"eta_decision":eta_decision})
    import pandas as pd
    return pd.DataFrame(seq)

def generate_dataset(n_cases=320, cfg=CaseConfig()):
    import pandas as pd
    return pd.concat([generate_case(cfg, pid=i) for i in range(n_cases)], ignore_index=True)

df = generate_dataset()
print(df.shape)
df.head()


(11520, 11)


,pid,t,triage,complaint,hr,map,cap_icu,capacity_stale,action,eta_first_result,eta_decision
0,0,0,4,1,89,75,0,1,0,20,105
1,0,1,4,1,85,80,0,0,5,15,100
2,0,2,4,1,76,63,0,0,0,10,95
3,0,3,4,1,115,88,0,1,0,5,90
4,0,4,4,1,77,94,0,1,0,0,85


In [102]:

from torch.utils.data import Dataset, DataLoader
import numpy as np, torch

class SeqDataset(Dataset):
    def __init__(self, df, max_len=60):
        self.df = df; self.pids = df.pid.unique(); self.max_len = max_len
        self.feats = ["triage","complaint","hr","map","cap_icu","capacity_stale"]
        self.targets_reg = ["eta_first_result","eta_decision"]
    def __len__(self): return len(self.pids)
    def __getitem__(self, idx):
        pid = self.pids[idx]; sub = self.df[self.df.pid==pid].sort_values("t")
        X = sub[self.feats].values.astype(np.float32)
        y_cls = int(sub.iloc[-1]["action"])
        y_reg = sub[self.targets_reg].iloc[-1].values.astype(np.float32)
        length = X.shape[0]
        X = X[:self.max_len]
        pad_len = self.max_len - X.shape[0]
        if pad_len>0: X = np.vstack([X, np.zeros((pad_len, X.shape[1]), dtype=np.float32)])
        return torch.from_numpy(X), torch.tensor(y_cls), torch.from_numpy(y_reg), torch.tensor(min(length,self.max_len))

def collate(batch):
    X, y, r, L = zip(*batch)
    return torch.stack(X), torch.tensor(y), torch.stack(r), torch.stack(L)

pids = df.pid.unique(); np.random.shuffle(pids)
split = int(0.8*len(pids))
train_ids, val_ids = pids[:split], pids[split:]
train_df, val_df = df[df.pid.isin(train_ids)], df[df.pid.isin(val_ids)]
train_dl = DataLoader(SeqDataset(train_df), batch_size=64, shuffle=True, collate_fn=collate)
val_dl = DataLoader(SeqDataset(val_df), batch_size=64, shuffle=False, collate_fn=collate)
len(train_dl.dataset), len(val_dl.dataset)


(256, 64)

In [103]:

ACTIONS = ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"]

class OpsNet(nn.Module):
    def __init__(self, x_dim, hidden=128, n_actions=len(ACTIONS)):
        super().__init__()
        self.gru = nn.GRU(input_size=x_dim, hidden_size=hidden, num_layers=2, dropout=0.2, batch_first=True)
        self.post_gru_dropout = nn.Dropout(0.2)
        self.cls = nn.Sequential(nn.Linear(hidden,64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, n_actions))
        self.reg = nn.Sequential(nn.Linear(hidden,64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 2), nn.Softplus())
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out,_ = self.gru(packed)
        out,_ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        idx = (lengths-1).view(-1,1,1).expand(-1,1,out.size(-1))
        last = out.gather(1, idx).squeeze(1)
        last = self.post_gru_dropout(last)
        return self.cls(last), self.reg(last)

def step_loop(model, dl, opt=None):
    model.train(mode=opt is not None)
    ce = nn.CrossEntropyLoss(label_smoothing=0.05)
    l1 = nn.L1Loss()
    total=acc=0; loss_sum=mae_sum=0.0
    for X,y,r,L in dl:
        X,y,r,L = X.to(device), y.to(device), r.to(device), L.to(device)
        logits, preds = model(X,L)
        loss = ce(logits,y) + 0.2*l1(preds,r)
        if opt:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        loss_sum += loss.item()*X.size(0)
        acc += (logits.argmax(1)==y).sum().item()
        mae_sum += torch.abs(preds-r).mean().item()*X.size(0)
        total += X.size(0)
    return loss_sum/total, acc/total, mae_sum/total

model = OpsNet(x_dim=6).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

EPOCHS=6
for e in range(EPOCHS):
    tl,ta,tmae = step_loop(model, train_dl, opt)
    vl,va,vmae = step_loop(model, val_dl, None)
    print(f"epoch {e+1}/{EPOCHS}  train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}  ETA MAE {vmae:.1f}m")


epoch 1/6  train 1.874/0.711  val 1.381/0.938  ETA MAE 0.4m
epoch 2/6  train 1.137/0.957  val 0.759/0.938  ETA MAE 0.2m
epoch 3/6  train 0.638/0.957  val 0.564/0.938  ETA MAE 0.1m
epoch 4/6  train 0.492/0.957  val 0.584/0.938  ETA MAE 0.0m
epoch 5/6  train 0.516/0.957  val 0.601/0.938  ETA MAE 0.0m
epoch 6/6  train 0.522/0.957  val 0.569/0.938  ETA MAE 0.0m


In [104]:

def propose_next_action(model, seq_df):
    feats = ["triage","complaint","hr","map","cap_icu","capacity_stale"]
    X = torch.from_numpy(seq_df[feats].values.astype(np.float32)).unsqueeze(0).to(device)
    L = torch.tensor([X.shape[1]], dtype=torch.long).to(device)
    logits, reg = model(X, L)
    action_id = int(logits.argmax(1).item())
    etas = reg.detach().cpu().numpy()[0].tolist()
    etas = [max(0.0, float(x)) for x in etas]
    return ACTIONS[action_id], {"eta_first_result_min": etas[0], "eta_decision_min": etas[1]}

pid = int(val_df.pid.sample(1).iloc[0])
sub = val_df[val_df.pid==pid].sort_values("t").reset_index(drop=True)
action, meta = propose_next_action(model, sub.iloc[:20])
print("PID", pid, "→ propose:", action, meta)


PID 27 → propose: none {'eta_first_result_min': 0.0035583125427365303, 'eta_decision_min': 0.002680440666154027}


In [124]:

from time import sleep

ACTION_TO_CODE = {
    "place_labs": "TROPONIN^Troponin I",
    "place_xray": "CXR^Chest X-ray",
    "place_ct":   "CT_HEAD^CT Head"
}
def order_bridge(patient_id: str, proposed_action: str, delay_sec=10):
    code = ACTION_TO_CODE.get(proposed_action)
    if not code:
        print(f"[BRIDGE] No ORM mapping for action '{proposed_action}'. Skipping.")
        return None
    path = orbis_sim.place_order(patient_id, code, delay_sec=delay_sec)
    print(f"[BRIDGE] Placed {code} for {patient_id} -> {path}")
    return path

def replay_loop(rn, cycles=5, sleep_sec=2):
    for i in range(cycles):
        info = orbis_sim.process_inbox_and_pending()
        processed = orbis_sim.pump_to_notifier(rn)
        print(f"[REPLAY] cycle {i+1}/{cycles} | moved={len(info['moved_inbox'])} oru={len(info['emitted_oru'])} pushed={len(processed)}")
        sleep(sleep_sec)
# --- Add BED REQUEST support to the ORBIS bridge ---
import json, time
from pathlib import Path

def _write_bedreq(base_dir, pid, level="WARD"):
    ts = time.strftime("%Y%m%d%H%M%S")
    pkt = {"msg_type":"BED_REQUEST","patient_id":pid,"level":level,
           "requested_at":ts,"status":"NEW"}
    out = Path(base_dir)/"orbis_sim"/"inbox"/f"{ts}_BEDREQ_{pid}_{level}.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(pkt))
    print(f"[BRIDGE] Bed request {level} for {pid} -> {out}")

# add to your order_bridge() logic:
def order_bridge(pid, action, delay_sec=2, base_dir=str(BASE)):
    # ... your existing mappings ...
    if action == "prepare_admit_ward":
        _write_bedreq(base_dir, pid, "WARD"); return
    if action == "prepare_admit_icu":
        _write_bedreq(base_dir, pid, "ICU");  return
    if action in ("none", None):
        print("[BRIDGE] No ORM mapping for 'none'. Skipping."); return
    # ... rest of your bridge for labs/xray/ct/pages ...

# Demo E2E
pat = "P-DEMO-1"
orbis_sim.admit_patient(pat)
action, meta = propose_next_action(model, sub.iloc[:20])
print("[ACTION]", pat, action, meta)
order_bridge(pat, action, delay_sec=3)
replay_loop(rn, cycles=5, sleep_sec=1)

# Reminder demo: force overdue
re.ingest_event(pat, "arrival", {})
re.apply_lab_protocol(pat, {"test_name":"troponin","initial_order_due_min":10,"repeat_schedule_min":[60,180]})
re.patients[pat]["timers"][0]["due_at"] = re.now_fn()
re.ingest_event(pat, "heartbeat", {})


[ACTION] P-DEMO-1 none {'eta_first_result_min': 0.003541354788467288, 'eta_decision_min': 0.002677368698641658}
[BRIDGE] No ORM mapping for 'none'. Skipping.
[REPLAY] cycle 1/5 | moved=0 oru=0 pushed=1
[REPLAY] cycle 2/5 | moved=0 oru=0 pushed=0
[REPLAY] cycle 3/5 | moved=0 oru=0 pushed=0
[REPLAY] cycle 4/5 | moved=0 oru=0 pushed=0
[REPLAY] cycle 5/5 | moved=0 oru=0 pushed=0


In [106]:
# --- Actionable proposal with top-k fallback + nicer ETA display ---

import numpy as np
import torch

ACTIONS = ["none","place_labs","place_xray","place_ct",
           "page_cardiology","page_neurology",
           "prepare_admit_ward","prepare_admit_icu"]

# what we can actually order via ORBIS in this PoC
ACTION_TO_CODE = {
    "place_labs": "TROPONIN^Troponin I",
    "place_xray": "CXR^Chest X-ray",
    "place_ct":   "CT_HEAD^CT Head",
}

def propose_topk(model, seq_df, k=3):
    feats = ["triage","complaint","hr","map","cap_icu","capacity_stale"]
    X = torch.from_numpy(seq_df[feats].values.astype(np.float32)).unsqueeze(0).to(device)
    L = torch.tensor([X.shape[1]], dtype=torch.long).to(device)
    logits, reg = model(X, L)
    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
    order = np.argsort(-probs)
    # ETAs (clamped to >=0)
    etas = reg.detach().cpu().numpy()[0].tolist()
    etas = [max(0.0, float(x)) for x in etas]
    topk = [(ACTIONS[i], float(probs[i])) for i in order[:k]]
    return topk, {"eta_first_result_min": round(etas[0], 2), "eta_decision_min": round(etas[1], 2)}

def choose_actionable(topk):
    """Pick the best action we can actually translate to an ORM; fall back from 'none'."""
    for name, p in topk:
        if name in ACTION_TO_CODE:
            return name, p
    # nothing actionable in top-k
    return "none", topk[0][1]

def order_bridge(patient_id: str, proposed_action: str, delay_sec=10):
    code = ACTION_TO_CODE.get(proposed_action)
    if not code:
        print(f"[BRIDGE] No ORM mapping for '{proposed_action}'. Skipping.")
        return None
    path = orbis_sim.place_order(patient_id, code, delay_sec=delay_sec)
    print(f"[BRIDGE] Placed {code} for {patient_id} -> {path}")
    return path

from time import sleep
def replay_loop(rn, cycles=5, sleep_sec=2):
    for i in range(cycles):
        info = orbis_sim.process_inbox_and_pending()
        pushed = orbis_sim.pump_to_notifier(rn)
        print(f"[REPLAY] {i+1}/{cycles} | moved={len(info['moved_inbox'])} oru={len(info['emitted_oru'])} pushed={len(pushed)}")
        sleep(sleep_sec)

# --- Demo run that won't stall on 'none' ---
pid_demo = "P-DEMO-1"
orbis_sim.admit_patient(pid_demo)

# pick a validation sequence; use first ~20 timesteps as the current context
sample_pid = int(val_df.pid.sample(1).iloc[0])
sub = val_df[val_df.pid==sample_pid].sort_values("t").reset_index(drop=True)

topk, meta = propose_topk(model, sub.iloc[:20], k=4)
print("Top-k:", topk, "| ETAs:", meta)

action, conf = choose_actionable(topk)
print("[ACTION]", pid_demo, action, f"(p={conf:.2f})", meta)

order_bridge(pid_demo, action, delay_sec=3)
replay_loop(rn, cycles=5, sleep_sec=1)

# Reminder demo (force overdue once so you see the alert)
re.ingest_event(pid_demo, "arrival", {})
re.apply_lab_protocol(pid_demo, {"test_name":"troponin","initial_order_due_min":10,"repeat_schedule_min":[60,180]})
re.patients[pid_demo]["timers"][0]["due_at"] = re.now_fn()
re.ingest_event(pid_demo, "heartbeat", {})


Top-k: [('none', 0.9606958627700806), ('page_neurology', 0.007640796713531017), ('prepare_admit_ward', 0.005668151658028364), ('place_xray', 0.00561506999656558)] | ETAs: {'eta_first_result_min': 0.0, 'eta_decision_min': 0.0}
[ACTION] P-DEMO-1 place_xray (p=0.01) {'eta_first_result_min': 0.0, 'eta_decision_min': 0.0}
[BRIDGE] Placed CXR^Chest X-ray for P-DEMO-1 -> ed_demo_data/orbis_sim/inbox/20250812131519_ORM_O01_P-DEMO-1_PLC1755004519.hl7
[REPLAY] 1/5 | moved=1 oru=0 pushed=1
[REPLAY] 2/5 | moved=0 oru=0 pushed=0
[REPLAY] 3/5 | moved=0 oru=0 pushed=0
[REPLAY] 4/5 | moved=0 oru=1 pushed=1
[REPLAY] 5/5 | moved=0 oru=0 pushed=0


In [107]:
# Replace your dataset with this
from torch.utils.data import Dataset
import numpy as np, torch

ACTIONS = ["none","place_labs","place_xray","place_ct",
           "page_cardiology","page_neurology",
           "prepare_admit_ward","prepare_admit_icu"]

class SeqNextDataset(Dataset):
    def __init__(self, df, max_len=60, min_ctx=5):
        self.df = df; self.pids = df.pid.unique(); self.max_len=max_len; self.min_ctx=min_ctx
        self.feats = ["triage","complaint","hr","map","cap_icu","capacity_stale"]
        self.targets_reg = ["eta_first_result","eta_decision"]
    def __len__(self): return len(self.pids)
    def __getitem__(self, idx):
        pid = self.pids[idx]
        sub = self.df[self.df.pid==pid].sort_values("t").reset_index(drop=True)
        T = len(sub)
        cut_lo = max(1, self.min_ctx)
        cut_hi = max(cut_lo+1, T-2)  # ensure there's a next step
        cut = np.random.randint(cut_lo, cut_hi) if cut_hi>cut_lo else cut_lo
        X = sub.loc[:cut, self.feats].values.astype(np.float32)         # context up to cut (inclusive)
        y_cls = int(sub.loc[cut+1, "action"])                           # NEXT action
        y_reg = sub.loc[cut, self.targets_reg].values.astype(np.float32)# ETAs from current cut
        length = X.shape[0]
        X = X[:self.max_len]
        pad = self.max_len - X.shape[0]
        if pad>0: X = np.vstack([X, np.zeros((pad, X.shape[1]), np.float32)])
        return torch.from_numpy(X), torch.tensor(y_cls), torch.from_numpy(y_reg), torch.tensor(min(length,self.max_len))

def collate(batch):
    X,y,r,L = zip(*batch)
    return torch.stack(X), torch.tensor(y), torch.stack(r), torch.stack(L)

train_dl = DataLoader(SeqNextDataset(train_df), batch_size=64, shuffle=True, collate_fn=collate)
val_dl   = DataLoader(SeqNextDataset(val_df),   batch_size=64, shuffle=False, collate_fn=collate)


In [108]:
# Estimate class weights from the SeqNextDataset
tmp_ds = SeqNextDataset(train_df)
labels = []
for i in range(len(tmp_ds)):
    _, y, _, _ = tmp_ds[i]
    labels.append(int(y))
import numpy as np, torch
counts = np.bincount(np.array(labels), minlength=len(ACTIONS))
weights = counts.sum() / np.maximum(counts, 1)
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
print("class counts:", counts, "\nclass weights:", weights)

# Plug into your training loop
ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
# keep L1 for ETAs as you had


class counts: [221   3   6   6   0  12   7   1] 
class weights: [0.01249373 0.92037109 0.46018555 0.46018555 2.76111328 0.23009277
 0.39444475 2.76111328]


In [109]:
# Force a valid probability vector (quick fix)
import importlib, sys
sys.path.insert(0, ".")
sg = importlib.import_module("synthetic_generator")

cfg = sg.default_config()
cfg["arrival_process"]["chest_pain_frac"] = 0.30  # must be ≤ 1 - (0.70 for the other complaints)

tables, metrics = sg.generate_day(cfg, seed=123)
print("Metrics:", metrics)

events_syn = tables["events"].copy()
events_syn.head()


Metrics: {'n_encounters': 286, 'labs_tat_p50': 82.33970785833333, 'labs_tat_p90': 195.4200428666667, 'cxr_tat_p50': 18.851658075, 'cxr_tat_p90': 34.87945380833333, 'ct_tat_p50': 25.13752023333333, 'ct_tat_p90': 59.19565990333334}


,enc_id,patient_id,t,triage,complaint,hr,map,cap_icu,capacity_stale,action,eta_first_result,eta_decision
0,E-00001,P-00001,0,3,0,78,79,1,0,2,32.293887,360.0
1,E-00001,P-00001,5,3,0,76,77,1,0,2,27.293887,355.0
2,E-00001,P-00001,10,3,0,85,75,1,0,2,22.293887,350.0
3,E-00001,P-00001,15,3,0,89,83,1,0,2,17.293887,345.0
4,E-00001,P-00001,20,3,0,85,86,1,0,0,12.293887,340.0


In [110]:
# STEP C — adapt columns (non-destructive: *_syn variables)
import pandas as pd
events_syn = tables["events"].copy()

# add numeric pid per encounter and order by time
events_syn["pid"] = pd.factorize(events_syn["enc_id"])[0].astype(int)
events_syn = events_syn[
    ["pid","t","triage","complaint","hr","map","cap_icu","capacity_stale",
     "action","eta_first_result","eta_decision"]
].sort_values(["pid","t"]).reset_index(drop=True)

print(events_syn.shape)
events_syn.head()

# sanity: class balance
import numpy as np
counts = np.bincount(events_syn["action"].values, minlength=8)
print("class counts:", counts)


(16798, 11)
class counts: [14864   264   315    77   118   241   608   311]


In [111]:
# STEP D — synthetic train/val splits + dataloaders
from torch.utils.data import DataLoader
rng = np.random.default_rng(123)

pids_all = events_syn.pid.unique()
rng.shuffle(pids_all)
split = int(0.8 * len(pids_all))
train_ids_syn, val_ids_syn = pids_all[:split], pids_all[split:]

train_df_syn = events_syn[events_syn.pid.isin(train_ids_syn)]
val_df_syn   = events_syn[events_syn.pid.isin(val_ids_syn)]

# reuse your existing SeqDataset / collate from the notebook
train_dl_syn = DataLoader(SeqDataset(train_df_syn), batch_size=64, shuffle=True, collate_fn=collate)
val_dl_syn   = DataLoader(SeqDataset(val_df_syn),   batch_size=64, shuffle=False, collate_fn=collate)

len(train_dl_syn.dataset), len(val_dl_syn.dataset)


(228, 58)

In [112]:
# STEP E — train an isolated model on synthetic data
model_syn = OpsNet(x_dim=6).to(device)
opt_syn   = torch.optim.AdamW(model_syn.parameters(), lr=1e-3)

EPOCHS = 6
for e in range(EPOCHS):
    tl,ta,tmae = step_loop(model_syn, train_dl_syn, opt_syn)
    vl,va,vmae = step_loop(model_syn, val_dl_syn, None)
    print(f"[SYN] epoch {e+1}/{EPOCHS}  train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}  ETA MAE {vmae:.1f}m")


[SYN] epoch 1/6  train 2.315/0.439  val 1.907/0.638  ETA MAE 2.3m
[SYN] epoch 2/6  train 1.682/0.667  val 1.484/0.638  ETA MAE 2.1m
[SYN] epoch 3/6  train 1.381/0.667  val 1.350/0.638  ETA MAE 1.9m
[SYN] epoch 4/6  train 1.266/0.667  val 1.285/0.638  ETA MAE 1.8m
[SYN] epoch 5/6  train 1.208/0.671  val 1.274/0.638  ETA MAE 1.8m
[SYN] epoch 6/6  train 1.219/0.601  val 1.273/0.638  ETA MAE 1.7m


In [113]:
# STEP F — write HL7 to your existing BASE folders and pump notifier
n_msgs = len(emit_hl7(tables, base_dir=str(BASE), write_orders=True, write_results=True))
print("HL7 messages written:", n_msgs)

# your existing replay loop + rn/orbis_sim from earlier cells
replay_loop(rn, cycles=6, sleep_sec=1)


HL7 messages written: 668
[REPLAY] 1/6 | moved=50 oru=0 pushed=100
[REPLAY] 2/6 | moved=50 oru=0 pushed=100
[REPLAY] 3/6 | moved=50 oru=0 pushed=100
[REPLAY] 4/6 | moved=41 oru=0 pushed=100
[REPLAY] 5/6 | moved=0 oru=0 pushed=77
[REPLAY] 6/6 | moved=0 oru=0 pushed=0


In [114]:
# STEP G — propose on synthetic context and place an order
# pick a synthetic validation sequence
sample_pid = int(val_df_syn.pid.sample(1).iloc[0])
sub_syn = val_df_syn[val_df_syn.pid==sample_pid].sort_values("t").reset_index(drop=True)

# prefer your top-k helper; fall back if missing
try:
    topk, meta = propose_topk(model_syn, sub_syn.iloc[:20], k=4)
    print("Top-k:", topk, "| ETAs:", meta)
    action, conf = choose_actionable(topk)  # from your earlier helper
    print("[ACTION]", "P-DEMO-SYN", action, f"(p={conf:.2f})")
except NameError:
    action, meta = propose_next_action(model_syn, sub_syn.iloc[:20])  # your earlier helper
    print("[ACTION]", "P-DEMO-SYN", action, meta)

# admit a demo patient, place the order, and replay
orbis_sim.admit_patient("P-DEMO-SYN")
order_bridge("P-DEMO-SYN", action, delay_sec=3)   # uses your ACTION_TO_CODE mapping
replay_loop(rn, cycles=6, sleep_sec=1)


Top-k: [('prepare_admit_ward', 0.5905312299728394), ('prepare_admit_icu', 0.3846736550331116), ('place_xray', 0.008250826969742775), ('page_neurology', 0.004042579792439938)] | ETAs: {'eta_first_result_min': 0.02, 'eta_decision_min': 0.01}
[ACTION] P-DEMO-SYN place_xray (p=0.01)
[BRIDGE] Placed CXR^Chest X-ray for P-DEMO-SYN -> ed_demo_data/orbis_sim/inbox/20250812131541_ORM_O01_P-DEMO-SYN_PLC1755004541.hl7
[REPLAY] 1/6 | moved=1 oru=0 pushed=1
[REPLAY] 2/6 | moved=0 oru=0 pushed=0
[REPLAY] 3/6 | moved=0 oru=0 pushed=0
[REPLAY] 4/6 | moved=0 oru=1 pushed=1
[REPLAY] 5/6 | moved=0 oru=0 pushed=0
[REPLAY] 6/6 | moved=0 oru=0 pushed=0


In [115]:
# sanity: do we already have the data from a successful generate?
print("tables in memory:", "tables" in globals())


tables in memory: True


In [116]:
# === TRAIN-READY PACK (non-destructive; uses _syn2 names) ===
# Assumes `tables` is already in memory with tables["events"] (from your synthetic day)
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader
from dataclasses import dataclass
from collections import defaultdict
from sklearn.metrics import balanced_accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 0) Prepare events (rebuild safely and keep enc_id for temporal split) ----------
events_syn2 = tables["events"].copy()
# numeric pid per encounter; keep enc_id for temporal split by arrival_ts
pid_codes, pid_uniques = pd.factorize(events_syn2["enc_id"])
events_syn2["pid"] = pid_codes.astype(int)

# restrict to the feature set your trainer uses
events_syn2 = events_syn2.sort_values(["pid","t"]).reset_index(drop=True)
events_syn2 = events_syn2[
    ["enc_id","pid","t","triage","complaint","hr","map","cap_icu","capacity_stale",
     "action","eta_first_result","eta_decision"]
]

# ---------- 1) Relabel to NEXT meaningful action within horizon ----------
HORIZON_MIN = 40  # tune 30–45

def relabel_next_action(df: pd.DataFrame, horizon_min: int) -> pd.DataFrame:
    # For each pid, at each time t, label = first non-"none" action within [t, t+horizon]
    # gate = 1 if such action exists, else 0
    rows = []
    for pid, g in df.groupby("pid"):
        g = g.sort_values("t").reset_index(drop=True)
        t_vals = g["t"].to_numpy()
        a_vals = g["action"].to_numpy()
        eta1   = g["eta_first_result"].to_numpy()
        eta2   = g["eta_decision"].to_numpy()
        x_mat  = g[["triage","complaint","hr","map","cap_icu","capacity_stale"]].to_numpy()

        y_next = np.zeros_like(a_vals)  # default none
        y_gate = np.zeros_like(a_vals)
        for i in range(len(g)):
            t0 = t_vals[i]
            # scan forward until horizon
            y = 0
            for j in range(i, len(g)):
                if t_vals[j] - t0 > horizon_min:  # out of horizon
                    break
                if a_vals[j] != 0:
                    y = a_vals[j]
                    y_gate[i] = 1
                    break
            y_next[i] = y

        block = pd.DataFrame({
            "pid": pid,
            "t": t_vals,
            "triage": x_mat[:,0], "complaint": x_mat[:,1],
            "hr": x_mat[:,2], "map": x_mat[:,3],
            "cap_icu": x_mat[:,4], "capacity_stale": x_mat[:,5],
            "y_action": y_next.astype(int),
            "y_gate": y_gate.astype(int),
            "eta_first_result": eta1,
            "eta_decision": eta2
        })
        rows.append(block)
    out = pd.concat(rows, ignore_index=True)
    return out.sort_values(["pid","t"]).reset_index(drop=True)

train_df_syn2_full = relabel_next_action(events_syn2, HORIZON_MIN)

# ---------- 2) Temporal split by encounter arrival time ----------
# Map enc_id -> arrival_ts from tables["encounters"], then pid -> arrival_ts via first enc_id per pid
enc = tables["encounters"][["enc_id","arrival_ts"]].copy()
pid_first_enc = events_syn2.groupby("pid").first().reset_index()[["pid","enc_id"]]
pid_ts = pid_first_enc.merge(enc, on="enc_id", how="left")
pid_ts["arrival_ts"] = pd.to_datetime(pid_ts["arrival_ts"])
pid_ts = pid_ts.sort_values("arrival_ts")
pids_sorted = pid_ts["pid"].to_numpy()

cut = int(math.floor(0.8 * len(pids_sorted)))
train_pids_syn2 = pids_sorted[:cut]
val_pids_syn2   = pids_sorted[cut:]

train_df_syn2 = train_df_syn2_full[train_df_syn2_full.pid.isin(train_pids_syn2)].copy()
val_df_syn2   = train_df_syn2_full[train_df_syn2_full.pid.isin(val_pids_syn2)].copy()

# ---------- 3) Dataset + Collate (independent of your older SeqDataset) ----------
X_COLS = ["triage","complaint","hr","map","cap_icu","capacity_stale"]
Y_ACTIONS = 8  # ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"]

def group_by_pid(df):
    for pid, g in df.groupby("pid"):
        g = g.sort_values("t")
        x = g[X_COLS].to_numpy(dtype=np.float32)
        y_gate   = g["y_gate"].to_numpy(dtype=np.float32)
        y_action = g["y_action"].to_numpy(dtype=np.int64)
        eta = g[["eta_first_result","eta_decision"]].to_numpy(dtype=np.float32)
        yield pid, x, y_gate, y_action, eta

class SeqSet:
    def __init__(self, df):
        self.pk = []
        self.data = []
        for pid, x, yg, ya, eta in group_by_pid(df):
            self.pk.append(pid)
            self.data.append((x, yg, ya, eta))
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate_pad(batch):
    # batch: list of (x[T,d], y_gate[T], y_action[T], eta[T,2])
    lens = [b[0].shape[0] for b in batch]
    T = max(lens)
    d = batch[0][0].shape[1]
    B = len(batch)
    x = torch.zeros(B, T, d, dtype=torch.float32)
    y_gate = torch.zeros(B, T, dtype=torch.float32)
    y_action = torch.zeros(B, T, dtype=torch.long)
    eta = torch.zeros(B, T, 2, dtype=torch.float32)
    mask = torch.zeros(B, T, dtype=torch.bool)
    for i,(xi, yi_g, yi_a, ei) in enumerate(batch):
        L = xi.shape[0]
        x[i,:L,:] = torch.from_numpy(xi)
        y_gate[i,:L] = torch.from_numpy(yi_g)
        y_action[i,:L] = torch.from_numpy(yi_a)
        eta[i,:L,:] = torch.from_numpy(ei)
        mask[i,:L] = True
    return x.to(device), y_gate.to(device), y_action.to(device), eta.to(device), mask.to(device)

train_dl_syn2 = DataLoader(SeqSet(train_df_syn2), batch_size=64, shuffle=True, collate_fn=collate_pad)
val_dl_syn2   = DataLoader(SeqSet(val_df_syn2),   batch_size=64, shuffle=False, collate_fn=collate_pad)

# ---------- 4) Model: GRU backbone + 3 heads (gate/action/ETAs) ----------
class GateActionETANet(nn.Module):
    def __init__(self, x_dim=6, hidden=64, n_actions=Y_ACTIONS):
        super().__init__()
        self.gru = nn.GRU(x_dim, hidden, num_layers=1, batch_first=True)
        self.head_gate = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 1))  # logits
        self.head_act  = nn.Sequential(nn.Linear(hidden, 128), nn.ReLU(), nn.Linear(128, n_actions))
        self.head_eta  = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 2))  # raw -> softplus
        self.softplus  = nn.Softplus(beta=1.0)
    def forward(self, x, mask):
        # x: (B,T,D)
        h, _ = self.gru(x)  # (B,T,H)
        gate_logits = self.head_gate(h).squeeze(-1)         # (B,T)
        act_logits  = self.head_act(h)                      # (B,T,C)
        eta_raw     = self.head_eta(h)                      # (B,T,2)
        eta_pred    = self.softplus(eta_raw) + 1e-3         # enforce non-negative ETA
        # mask positions outside sequence
        gate_logits = gate_logits.masked_fill(~mask, 0.0)
        act_logits  = act_logits.masked_fill(~mask.unsqueeze(-1), 0.0)
        eta_pred    = eta_pred.masked_fill(~mask.unsqueeze(-1), 0.0)
        return gate_logits, act_logits, eta_pred

model_syn2 = GateActionETANet(x_dim=len(X_COLS)).to(device)

# ---------- 5) Losses with imbalance control ----------
# Compute class stats on TRAIN ONLY
pos_count = int((train_df_syn2["y_gate"]==1).sum())
neg_count = int((train_df_syn2["y_gate"]==0).sum())
pos_weight = torch.tensor([(neg_count / max(1,pos_count))], dtype=torch.float32, device=device)  # >1 if positives are rare

# action weights among positive samples only
pos_actions = train_df_syn2.loc[train_df_syn2["y_gate"]==1, "y_action"].to_numpy()
act_counts = np.bincount(pos_actions, minlength=Y_ACTIONS)  # includes index 0 but gate excludes it
# avoid zero for unseen classes
act_weights = np.where(act_counts>0, act_counts.sum()/np.maximum(1, act_counts), 1.0).astype(np.float32)
act_weights[0] = 0.0  # "none" is never trained in action head (gate=0 then action ignored)
w_action = torch.tensor(act_weights, device=device)

bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
ce  = nn.CrossEntropyLoss(weight=w_action, reduction="sum")  # we'll divide by n_pos later
l1  = nn.L1Loss(reduction="sum")  # we'll normalize by #masked positions

def step_epoch(model, loader, opt=None):
    model.train(True) if opt else model.train(False)
    tot_loss = tot_gate = tot_act = tot_eta = 0.0
    n_pos = 0
    n_mask = 0
    # metrics accumulators
    y_true, y_pred = [], []
    eta_abs_err = []

    for x, y_gate, y_action, eta, mask in loader:
        gate_logits, act_logits, eta_pred = model(x, mask)
        # gate loss over all masked positions
        gate_loss = bce(gate_logits[mask], y_gate[mask])

        # action loss only where gate target is 1
        pos_mask = (y_gate > 0.5) & mask
        if pos_mask.any():
            ce_loss = ce(act_logits[pos_mask], y_action[pos_mask])
            ce_loss = ce_loss / (pos_mask.sum().item())  # mean over positives
        else:
            ce_loss = torch.tensor(0.0, device=device)

        # eta loss over all masked positions (two heads)
        eta_loss = l1(eta_pred[mask], eta[mask]) / (mask.sum().item())

        loss = gate_loss + ce_loss + 0.1 * eta_loss

        if opt:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # accumulate
        tot_loss += loss.item()
        tot_gate += gate_loss.item()
        tot_act  += ce_loss.item() if isinstance(ce_loss, torch.Tensor) else ce_loss
        tot_eta  += eta_loss.item()
        n_pos    += int(pos_mask.sum().item())
        n_mask   += int(mask.sum().item())

        # metrics: on positives only
        if pos_mask.any():
            pred_top = act_logits[pos_mask].argmax(dim=-1).detach().cpu().numpy()
            y_pred.extend(pred_top.tolist())
            y_true.extend(y_action[pos_mask].detach().cpu().numpy().tolist())

        # eta MAE
        if mask.any():
            err = (eta_pred[mask] - eta[mask]).abs().detach().cpu().numpy()
            eta_abs_err.extend(err.reshape(-1,2).tolist())

    # derive metrics
    act_only_acc = float(np.mean(np.array(y_pred)==np.array(y_true))) if len(y_true) else float("nan")
    try:
        bal_acc = float(balanced_accuracy_score(y_true, y_pred)) if len(y_true) else float("nan")
    except Exception:
        bal_acc = float("nan")
    eta_mae = float(np.mean(np.abs(np.array(eta_abs_err)), axis=0).mean()) if len(eta_abs_err) else float("nan")

    denom = max(1, len(loader))
    return {
        "loss": tot_loss/denom, "gate": tot_gate/denom, "act": tot_act/denom, "eta": tot_eta/denom,
        "act_only_acc": act_only_acc, "bal_acc": bal_acc, "eta_mae_min": eta_mae, "n_pos": n_pos, "n_steps": n_mask
    }

# ---------- 6) Train ----------
opt2 = torch.optim.AdamW(model_syn2.parameters(), lr=1e-3)
EPOCHS = 8
for e in range(1, EPOCHS+1):
    tr = step_epoch(model_syn2, train_dl_syn2, opt=opt2)
    va = step_epoch(model_syn2, val_dl_syn2,   opt=None)
    print(f"[SYN2] epoch {e:02d}/{EPOCHS} | "
          f"train L={tr['loss']:.3f} (G {tr['gate']:.3f}/A {tr['act']:.3f}/E {tr['eta']:.3f}) "
          f"| val L={va['loss']:.3f} (G {va['gate']:.3f}/A {va['act']:.3f}/E {va['eta']:.3f}) "
          f"| act-only acc {va['act_only_acc']:.3f} bal {va['bal_acc']:.3f} | ETA MAE {va['eta_mae_min']:.1f}m "
          f"| n_pos={va['n_pos']}")

# ---------- 7) Inference helper (top-k with gate) ----------
ACTIONS = ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"]

@torch.no_grad()
def propose_topk_syn2(model, seq_df: pd.DataFrame, k=4, p_floor=0.10):
    seq = seq_df.sort_values("t")
    x = torch.from_numpy(seq[X_COLS].to_numpy(np.float32))[None,...].to(device)
    mask = torch.ones(1, x.shape[1], dtype=torch.bool, device=device)
    g_logit, a_logit, eta = model(x, mask)
    # take the last timestep
    g_p = torch.sigmoid(g_logit[0,-1]).item()
    a = torch.softmax(a_logit[0,-1], dim=-1).cpu().numpy()
    top_idx = np.argsort(a)[::-1][:k]
    top = [(ACTIONS[i], float(a[i])) for i in top_idx]
    # If the gate says "not now", override to 'none'
    if g_p < p_floor:
        return [("none", 1.0)] + top, {"gate_p": g_p, "eta_first_result_min": float(eta[0,-1,0].item()), "eta_decision_min": float(eta[0,-1,1].item())}
    return top, {"gate_p": g_p, "eta_first_result_min": float(eta[0,-1,0].item()), "eta_decision_min": float(eta[0,-1,1].item())}

print("\n[SYN2] Train-ready pack finished. Use `propose_topk_syn2(model_syn2, sub_df, k=4)` on a sequence.")


[SYN2] epoch 01/8 | train L=32.774 (G 1.079/A 14.505/E 171.899) | val L=32.738 (G 1.086/A 14.707/E 169.452) | act-only acc 0.134 bal 0.187 | ETA MAE 84.7m | n_pos=822
[SYN2] epoch 02/8 | train L=32.269 (G 1.089/A 14.203/E 169.781) | val L=32.405 (G 1.091/A 14.392/E 169.224) | act-only acc 0.414 bal 0.251 | ETA MAE 84.6m | n_pos=822
[SYN2] epoch 03/8 | train L=31.894 (G 1.075/A 13.729/E 170.911) | val L=32.114 (G 1.084/A 14.133/E 168.969) | act-only acc 0.468 bal 0.267 | ETA MAE 84.5m | n_pos=822
[SYN2] epoch 04/8 | train L=31.455 (G 1.072/A 13.368/E 170.156) | val L=31.830 (G 1.082/A 13.880/E 168.681) | act-only acc 0.483 bal 0.313 | ETA MAE 84.3m | n_pos=822
[SYN2] epoch 05/8 | train L=31.366 (G 1.077/A 13.370/E 169.188) | val L=31.473 (G 1.083/A 13.557/E 168.326) | act-only acc 0.517 bal 0.328 | ETA MAE 84.2m | n_pos=822
[SYN2] epoch 06/8 | train L=30.392 (G 1.061/A 12.285/E 170.454) | val L=31.092 (G 1.084/A 13.217/E 167.912) | act-only acc 0.478 bal 0.361 | ETA MAE 84.0m | n_pos=82

In [117]:
# --- PATCH: better ETA loss + gate calibration + top-k eval (non-destructive) ---
import numpy as np, torch, torch.nn as nn
from sklearn.metrics import precision_recall_curve, f1_score

# 1) New step with log1p ETA loss (drop-in for validation & fine-tune)
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.0], device=device))  # pos_weight recomputed below
ce  = nn.CrossEntropyLoss(reduction="sum")
l1  = nn.L1Loss(reduction="sum")

def _class_weights_from_train(df, n_actions=8):
    pos = df.loc[df["y_gate"]==1, "y_action"].to_numpy()
    counts = np.bincount(pos, minlength=n_actions)
    w = np.where(counts>0, counts.sum()/np.maximum(1, counts), 1.0).astype(np.float32)
    w[0] = 0.0  # never train "none" in action head
    return torch.tensor(w, device=device)

def step_epoch_v2(model, loader, train_df, opt=None):
    # compute weights fresh (robust if splits change)
    pos_count = int((train_df["y_gate"]==1).sum())
    neg_count = int((train_df["y_gate"]==0).sum())
    bce.pos_weight = torch.tensor([(neg_count/max(1,pos_count))], device=device)
    ce.weight = _class_weights_from_train(train_df, n_actions=8)

    model.train(bool(opt))
    tot = {"loss":0.0,"gate":0.0,"act":0.0,"eta":0.0}
    y_true, y_pred = [], []
    eta_abs_err = []
    n_batches = 0
    for x, y_gate, y_action, eta, mask in loader:
        g_logit, a_logit, eta_pred = model(x, mask)
        # --- losses ---
        gate_loss = bce(g_logit[mask], y_gate[mask])
        pos_mask = (y_gate > 0.5) & mask
        if pos_mask.any():
            act_loss = ce(a_logit[pos_mask], y_action[pos_mask]) / (pos_mask.sum().item())
        else:
            act_loss = torch.tensor(0.0, device=device)
        # log-space ETA loss
        eta_loss = l1(torch.log1p(eta_pred[mask]), torch.log1p(eta[mask])) / (mask.sum().item())
        loss = gate_loss + act_loss + 0.1*eta_loss

        if opt:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # accumulators
        for k,v in (("loss",loss),("gate",gate_loss),("act",act_loss),("eta",eta_loss)):
            tot[k] += float(v.item())
        n_batches += 1

        if pos_mask.any():
            y_pred.extend(a_logit[pos_mask].argmax(-1).detach().cpu().numpy().tolist())
            y_true.extend(y_action[pos_mask].detach().cpu().numpy().tolist())
        if mask.any():
            err = (eta_pred[mask] - eta[mask]).abs().detach().cpu().numpy()
            eta_abs_err.extend(err.reshape(-1,2).tolist())

    out = {k: v/max(1,n_batches) for k,v in tot.items()}
    out["act_only_acc"] = float(np.mean(np.array(y_pred)==np.array(y_true))) if len(y_true) else float("nan")
    out["eta_mae_min"]  = float(np.mean(np.abs(np.array(eta_abs_err)), axis=0).mean()) if len(eta_abs_err) else float("nan")
    return out

# 2) Fine-tune a few epochs with the new loss on your existing loaders
opt_ft = torch.optim.AdamW(model_syn2.parameters(), lr=7e-4)
for e in range(1, 5):
    tr = step_epoch_v2(model_syn2, train_dl_syn2, train_df_syn2, opt=opt_ft)
    va = step_epoch_v2(model_syn2, val_dl_syn2,   train_df_syn2, opt=None)
    print(f"[SYN2/logETA] e{e}/4 | train L={tr['loss']:.3f} (G {tr['gate']:.3f}/A {tr['act']:.3f}/E {tr['eta']:.3f}) "
          f"| val L={va['loss']:.3f} (G {va['gate']:.3f}/A {va['act']:.3f}/E {va['eta']:.3f}) "
          f"| act-only {va['act_only_acc']:.3f} | ETA MAE {va['eta_mae_min']:.1f}m")

# 3) Calibrate gate threshold on validation (maximize F1 for "act now")
@torch.no_grad()
def collect_gate_scores(model, loader):
    scores, labels = [], []
    for x, y_gate, y_action, eta, mask in loader:
        g_logit, a_logit, _ = model(x, mask)
        s = torch.sigmoid(g_logit[mask]).cpu().numpy()
        y = y_gate[mask].cpu().numpy()
        scores.extend(s.tolist()); labels.extend(y.tolist())
    return np.array(scores), np.array(labels)

scores, labels = collect_gate_scores(model_syn2, val_dl_syn2)
# avoid degenerate cases
if labels.sum() == 0 or labels.sum() == len(labels):
    gate_tau = 0.5
else:
    ps, rs, ts = precision_recall_curve(labels, scores)
    f1s = (2*ps*rs)/(ps+rs+1e-9)
    i = np.nanargmax(f1s)
    gate_tau = float(ts[i]) if i < len(ts) else 0.5
print(f"[CAL] gate threshold τ = {gate_tau:.2f} (PR-F1 maximized)")

# 4) Updated proposer using calibrated gate and top-k report
ACTIONS = ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"]

@torch.no_grad()
def propose_topk_syn2(model, seq_df, k=4, gate_tau_override=None):
    seq = seq_df.sort_values("t")
    x = torch.from_numpy(seq[["triage","complaint","hr","map","cap_icu","capacity_stale"]].to_numpy(np.float32))[None,...].to(device)
    mask = torch.ones(1, x.shape[1], dtype=torch.bool, device=device)
    g_logit, a_logit, eta = model(x, mask)
    g_p = torch.sigmoid(g_logit[0,-1]).item()
    a = torch.softmax(a_logit[0,-1], dim=-1).cpu().numpy()
    top_idx = np.argsort(a)[::-1][:k]
    top = [(ACTIONS[i], float(a[i])) for i in top_idx]
    tau = gate_tau if gate_tau_override is None else gate_tau_override
    meta = {"gate_p": g_p, "eta_first_result_min": float(eta[0,-1,0].item()), "eta_decision_min": float(eta[0,-1,1].item()), "tau": tau}
    if g_p < tau:
        return [("none", 1.0)] + top, meta
    return top, meta

# quick sanity: run on a random validation sequence
pid_demo = int(val_df_syn2.pid.sample(1, random_state=11).iloc[0])
sub = val_df_syn2[val_df_syn2.pid==pid_demo].sort_values("t").reset_index(drop=True)
topk, meta = propose_topk_syn2(model_syn2, sub.iloc[:20], k=4)
print("Top-k:", topk, "| meta:", meta)


[SYN2/logETA] e1/4 | train L=13.264 (G 1.060/A 11.830/E 3.743) | val L=13.612 (G 1.076/A 12.176/E 3.609) | act-only 0.523 | ETA MAE 83.2m
[SYN2/logETA] e2/4 | train L=12.777 (G 1.055/A 11.356/E 3.656) | val L=13.327 (G 1.074/A 11.900/E 3.520) | act-only 0.527 | ETA MAE 83.0m
[SYN2/logETA] e3/4 | train L=12.661 (G 1.060/A 11.242/E 3.592) | val L=13.081 (G 1.073/A 11.665/E 3.434) | act-only 0.517 | ETA MAE 82.8m
[SYN2/logETA] e4/4 | train L=12.378 (G 1.058/A 10.971/E 3.488) | val L=12.779 (G 1.070/A 11.373/E 3.360) | act-only 0.543 | ETA MAE 82.6m
[CAL] gate threshold τ = 0.49 (PR-F1 maximized)
Top-k: [('none', 1.0), ('prepare_admit_icu', 0.2644047737121582), ('prepare_admit_ward', 0.2594258785247803), ('page_cardiology', 0.15585607290267944), ('page_neurology', 0.11724153906106949)] | meta: {'gate_p': 0.463961660861969, 'eta_first_result_min': 0.013795692473649979, 'eta_decision_min': 4.669422626495361, 'tau': 0.49000978469848633}


In [118]:
# --- PATCH: clip ETA to 60 min + lower ETA weight; then recalibrate gate ---
import numpy as np, torch, torch.nn as nn
from sklearn.metrics import precision_recall_curve

ETA_CLIP = 60.0
ETA_W    = 0.03  # was 0.10

bce = nn.BCEWithLogitsLoss()
ce  = nn.CrossEntropyLoss(reduction="sum")
l1  = nn.L1Loss(reduction="sum")

def _class_weights_from_train(df, n_actions=8):
    pos = df.loc[df["y_gate"]==1, "y_action"].to_numpy()
    counts = np.bincount(pos, minlength=n_actions)
    w = np.where(counts>0, counts.sum()/np.maximum(1, counts), 1.0).astype(np.float32)
    w[0] = 0.0  # never train "none" in action head
    return torch.tensor(w, device=device)

def step_epoch_clip(model, loader, train_df, opt=None):
    # imbalance weights from TRAIN
    pos_count = int((train_df["y_gate"]==1).sum()); neg_count = int((train_df["y_gate"]==0).sum())
    bce.pos_weight = torch.tensor([(neg_count/max(1,pos_count))], device=device)
    ce.weight = _class_weights_from_train(train_df, n_actions=8)

    model.train(bool(opt))
    tot = {"loss":0.0,"gate":0.0,"act":0.0,"eta":0.0}
    y_true, y_pred, eta_abs_err = [], [], []
    n_batches = 0

    for x, y_gate, y_action, eta, mask in loader:
        # clip ETA targets
        eta_clip = torch.clamp(eta, min=0.0, max=ETA_CLIP)

        g_logit, a_logit, eta_pred = model(x, mask)
        # also clip predictions for the loss (keep non-negativity)
        eta_pred_clip = torch.clamp(eta_pred, min=0.0, max=ETA_CLIP)

        gate_loss = bce(g_logit[mask], y_gate[mask])

        pos_mask = (y_gate > 0.5) & mask
        if pos_mask.any():
            act_loss = ce(a_logit[pos_mask], y_action[pos_mask]) / (pos_mask.sum().item())
        else:
            act_loss = torch.tensor(0.0, device=device)

        eta_loss = l1(eta_pred_clip[mask], eta_clip[mask]) / (mask.sum().item())

        loss = gate_loss + act_loss + ETA_W*eta_loss

        if opt:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        for k,v in (("loss",loss),("gate",gate_loss),("act",act_loss),("eta",eta_loss)):
            tot[k] += float(v.item())
        n_batches += 1

        # metrics on positives + ETA MAE (unclipped, so we see real error)
        if pos_mask.any():
            y_pred.extend(a_logit[pos_mask].argmax(-1).detach().cpu().numpy().tolist())
            y_true.extend(y_action[pos_mask].detach().cpu().numpy().tolist())
        if mask.any():
            err = (eta_pred[mask] - eta[mask]).abs().detach().cpu().numpy()
            eta_abs_err.extend(err.reshape(-1,2).tolist())

    out = {k: v/max(1,n_batches) for k,v in tot.items()}
    out["act_only_acc"] = float(np.mean(np.array(y_pred)==np.array(y_true))) if len(y_true) else float("nan")
    out["eta_mae_min"]  = float(np.mean(np.abs(np.array(eta_abs_err)), axis=0).mean()) if len(eta_abs_err) else float("nan")
    return out

# fine-tune a few epochs
opt_ft2 = torch.optim.AdamW(model_syn2.parameters(), lr=6e-4)
for e in range(1, 4):
    tr = step_epoch_clip(model_syn2, train_dl_syn2, train_df_syn2, opt=opt_ft2)
    va = step_epoch_clip(model_syn2, val_dl_syn2,   train_df_syn2, opt=None)
    print(f"[SYN2/clip60] e{e}/3 | train L={tr['loss']:.3f} (G {tr['gate']:.3f}/A {tr['act']:.3f}/E {tr['eta']:.3f}) "
          f"| val L={va['loss']:.3f} (G {va['gate']:.3f}/A {va['act']:.3f}/E {va['eta']:.3f}) "
          f"| act-only {va['act_only_acc']:.3f} | ETA MAE {va['eta_mae_min']:.1f}m")

# re-calibrate gate τ after fine-tune
@torch.no_grad()
def collect_gate_scores(model, loader):
    scores, labels = [], []
    for x, y_gate, y_action, eta, mask in loader:
        g_logit, _, _ = model(x, mask)
        s = torch.sigmoid(g_logit[mask]).cpu().numpy()
        y = y_gate[mask].cpu().numpy()
        scores.extend(s.tolist()); labels.extend(y.tolist())
    return np.array(scores), np.array(labels)

scores, labels = collect_gate_scores(model_syn2, val_dl_syn2)
if labels.sum() in (0, len(labels)):
    gate_tau = 0.5
else:
    ps, rs, ts = precision_recall_curve(labels, scores)
    f1s = (2*ps*rs)/(ps+rs+1e-9)
    i = np.nanargmax(f1s)
    gate_tau = float(ts[i]) if i < len(ts) else 0.5
print(f"[CAL/clip60] gate τ = {gate_tau:.2f}")


[SYN2/clip60] e1/3 | train L=13.200 (G 1.050/A 10.623/E 50.891) | val L=13.666 (G 1.064/A 11.109/E 49.765) | act-only 0.524 | ETA MAE 82.5m
[SYN2/clip60] e2/3 | train L=12.961 (G 1.051/A 10.397/E 50.446) | val L=13.398 (G 1.060/A 10.856/E 49.382) | act-only 0.557 | ETA MAE 82.3m
[SYN2/clip60] e3/3 | train L=12.635 (G 1.037/A 10.088/E 50.335) | val L=13.128 (G 1.057/A 10.602/E 48.960) | act-only 0.625 | ETA MAE 82.1m
[CAL/clip60] gate τ = 0.52


In [125]:
# === REALISTIC TRAINING PACK (gate+action only, rule-based ETA for demo) ===
import numpy as np, pandas as pd, math, copy, torch, torch.nn as nn
from torch.utils.data import DataLoader
from datetime import datetime, timedelta
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- A) Generate 7 synthetic days with mild drift (uses synthetic_generator.py if present) ---
def _load_synth():
    from importlib.util import spec_from_file_location, module_from_spec
    for p in (Path("./synthetic_generator.py"), Path("/kaggle/working/synthetic_generator.py")):
        if p.exists():
            spec = spec_from_file_location("synthetic_generator", p)
            m = module_from_spec(spec); spec.loader.exec_module(m); return m
    raise FileNotFoundError("synthetic_generator.py not found. If kernel restarted, re-add it once.")

sg = _load_synth()

# PATCH: clamp chest_pain_frac so probs sum to 1 (generator hard-codes others=0.70)
def gen_day(start_dt: datetime, seed: int, drift: float=0.10, chest_frac: float=None):
    cfg = sg.default_config()
    cfg = copy.deepcopy(cfg)
    # drift TATs and ICU availability a bit
    for key in ("ct","xray","lab"):
        cfg["resources"]["service_times_min"][key]["scale"] *= float(np.random.uniform(1-drift, 1+drift))
    cfg["capacity"]["icu_free_init"] = int(round(cfg["capacity"]["icu_free_init"] * float(np.random.uniform(1-drift, 1+drift))))
    cfg["sampling"]["day_start"] = start_dt.strftime("%Y-%m-%d 00:00:00")

    # --- important: fix complaint probabilities ---
    OTHERS_SUM = 0.12 + 0.12 + 0.11 + 0.06 + 0.25 + 0.02 + 0.02  # = 0.70
    MAX_CHEST = 1.0 - OTHERS_SUM                                 # = 0.30
    c = cfg["arrival_process"].get("chest_pain_frac", 0.30) if chest_frac is None else float(chest_frac)
    cfg["arrival_process"]["chest_pain_frac"] = max(0.0, min(c, MAX_CHEST))  # clamp to [0, 0.30]

    tables, metrics = sg.generate_day(cfg, seed=seed)
    return tables, metrics

# regenerate the 7 days
days = []
start = datetime(2025,8,5)
for d in range(7):
    t, m = gen_day(start + timedelta(days=d), seed=123+d, drift=0.10, chest_frac=0.30)  # explicit 0.30
    t["events"]["day"] = d
    t["encounters"]["day"] = d
    days.append((t,m))

print("Generated days:", [m["n_encounters"] for _,m in days], "total encounters:", sum(m["n_encounters"] for _,m in days))

# --- B) Build train/val/test by day (0-4 train, 5 val, 6 test) and relabel to next-action within horizon ---
def relabel_next_action(events_df: pd.DataFrame, horizon_min: int=40) -> pd.DataFrame:
    out = []
    for pid, g in events_df.groupby(["day","enc_id"], sort=False):
        g = g.sort_values("t")
        t_vals = g["t"].to_numpy()
        a_vals = g["action"].to_numpy()
        y_next = np.zeros_like(a_vals); y_gate = np.zeros_like(a_vals)
        for i in range(len(g)):
            t0 = t_vals[i]; y = 0
            for j in range(i, len(g)):
                if t_vals[j] - t0 > horizon_min: break
                if a_vals[j] != 0: y = a_vals[j]; y_gate[i]=1; break
            y_next[i] = y
        block = g[["enc_id","t","triage","complaint","hr","map","cap_icu","capacity_stale","day"]].copy()
        block["y_action"] = y_next; block["y_gate"] = y_gate
        out.append(block)
    return pd.concat(out, ignore_index=True).sort_values(["day","enc_id","t"]).reset_index(drop=True)

events_all = pd.concat([t["events"] for t,_ in days], ignore_index=True)
enc_all    = pd.concat([t["encounters"] for t,_ in days], ignore_index=True)

# --- Build placed/pending features per (enc_id, t) ---
def build_order_features(rel_df, days):
    rel_df = rel_df.copy()
    rel_df["has_labs"]=0; rel_df["has_cxr"]=0; rel_df["has_ct"]=0
    rel_df["pend_labs"]=0; rel_df["pend_cxr"]=0; rel_df["pend_ct"]=0

    code_map = {
        "labs":   lambda s: "TROPONIN" in s,
        "cxr":    lambda s: "CXR" in s,
        "ct":     lambda s: "CT_" in s or "CT " in s,
    }

    # pre-index orders/results per day for speed
    ORD = {d: t["orders"].copy()   for d,(t,_) in enumerate(days)}
    RES = {d: t["results"].copy()  for d,(t,_) in enumerate(days)}
    for d in ORD: 
        if "ts_placed" in ORD[d]:  ORD[d]["ts_placed"] = pd.to_datetime(ORD[d]["ts_placed"])
        if "ts_result" in RES[d]:  RES[d]["ts_result"] = pd.to_datetime(RES[d]["ts_result"])

    def flags_for(enc_id, t_now_min, day):
        t_tbl, r_tbl = ORD[day], RES[day]
        # arrival ts (needed to convert mins since arrival to absolute time)
        arr = pd.to_datetime([tbl for tbl,_ in days][day]["encounters"].loc[
            [tbl for tbl,_ in days][day]["encounters"]["enc_id"]==enc_id, "arrival_ts"
        ].iloc[0])
        now = arr + pd.Timedelta(minutes=int(t_now_min))

        ords = t_tbl[t_tbl["enc_id"]==enc_id]
        ress = r_tbl[r_tbl["enc_id"]==enc_id]

        placed = {
            "labs": any(code_map["labs"](c) and ts <= now for c,ts in zip(ords["code"], ords["ts_placed"])),
            "cxr":  any(code_map["cxr"](c)  and ts <= now for c,ts in zip(ords["code"], ords["ts_placed"])),
            "ct":   any(code_map["ct"](c)   and ts <= now for c,ts in zip(ords["code"], ords["ts_placed"])),
        }
        # pending = placed but no result <= now
        def pending(kind):
            if not placed[kind]: return False
            # find accessions of that kind placed <= now
            accs = [acc for c,acc,ts in zip(ords["code"], ords["accession"], ords["ts_placed"])
                    if code_map[kind](c) and ts <= now]
            # any of those accessions missing a result by now?
            for acc in accs:
                got = any(a==acc and (not pd.isna(tsr)) and tsr <= now
                          for a,tsr in zip(ress["accession"], pd.to_datetime(ress["ts_result"])))
                if not got:
                    return True
            return False

        return placed["labs"], placed["cxr"], placed["ct"], pending("labs"), pending("cxr"), pending("ct")

    rel_df[["has_labs","has_cxr","has_ct","pend_labs","pend_cxr","pend_ct"]] = rel_df.apply(
        lambda r: pd.Series(flags_for(r["enc_id"], r["t"], int(r["day"]))), axis=1
    )
    return rel_df

# apply
rel_all_feat = build_order_features(rel_all, days)
print("Feature cols added. Sample:")
rel_all_feat[["enc_id","t","has_labs","pend_labs","has_cxr","pend_cxr","has_ct","pend_ct"]].head()


# numeric pid per enc_id per day
events_all["pid"] = pd.factorize(events_all["day"].astype(str) + "|" + events_all["enc_id"])[0].astype(int)
rel_all = relabel_next_action(events_all, horizon_min=40)

# features: add time-since-arrival normalized (helps gate/action timing)
rel_all = rel_all.merge(enc_all[["enc_id","arrival_ts","day"]], on=["enc_id","day"], how="left")
rel_all["arrival_ts"] = pd.to_datetime(rel_all["arrival_ts"])
rel_all["t_norm"] = rel_all["t"] / 360.0  # 6h window scaled
X_COLS = ["triage","complaint","hr","map","cap_icu","capacity_stale","t_norm"]

# splits
train_days, val_days, test_days = set(range(0,5)), {5}, {6}
train_df = rel_all[rel_all["day"].isin(train_days)].copy()
val_df   = rel_all[rel_all["day"].isin(val_days)].copy()
test_df  = rel_all[rel_all["day"].isin(test_days)].copy()

# --- C) Dataset/loader (gate+action only) ---
class SeqSet:
    def __init__(self, df):
        self.pks = []
        self.data = []
        for (d, enc), g in df.groupby(["day","enc_id"], sort=False):
            g = g.sort_values("t")
            x = g[X_COLS].to_numpy(np.float32)
            yg = g["y_gate"].to_numpy(np.float32)
            ya = g["y_action"].to_numpy(np.int64)
            self.pks.append((d,enc)); self.data.append((x,yg,ya))
    def __len__(self): return len(self.data)
    def __getitem__(self,i): return self.data[i]

def collate_pad(batch):
    lens = [b[0].shape[0] for b in batch]
    T = max(lens); B = len(batch); D = batch[0][0].shape[1]
    x = torch.zeros(B,T,D); yg = torch.zeros(B,T); ya = torch.zeros(B,T,dtype=torch.long); mask = torch.zeros(B,T,dtype=torch.bool)
    for i,(xi,gi,ai) in enumerate(batch):
        L=xi.shape[0]
        x[i,:L,:]=torch.from_numpy(xi); yg[i,:L]=torch.from_numpy(gi); ya[i,:L]=torch.from_numpy(ai); mask[i,:L]=True
    return x.to(device), yg.to(device), ya.to(device), mask.to(device)

train_dl = DataLoader(SeqSet(train_df), batch_size=64, shuffle=True,  collate_fn=collate_pad)
val_dl   = DataLoader(SeqSet(val_df),   batch_size=64, shuffle=False, collate_fn=collate_pad)
test_dl  = DataLoader(SeqSet(test_df),  batch_size=64, shuffle=False, collate_fn=collate_pad)

# --- D) Model: GRU + gate/action heads (no ETA head) ---
N_ACTIONS = 8
class GateActionNet(nn.Module):
    def __init__(self, x_dim):
        super().__init__()
        self.gru = nn.GRU(x_dim, 64, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,1))
        self.act  = nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,N_ACTIONS))
    def forward(self, x, mask):
        h,_ = self.gru(x)
        g = self.gate(h).squeeze(-1)
        a = self.act(h)
        g = g.masked_fill(~mask, 0.0); a = a.masked_fill(~mask.unsqueeze(-1), 0.0)
        return g,a

model_ga = GateActionNet(x_dim=len(X_COLS)).to(device)

# imbalance control
def action_weights(df):
    pos = df.loc[df["y_gate"]==1, "y_action"].to_numpy()
    cnt = np.bincount(pos, minlength=N_ACTIONS)
    w = np.where(cnt>0, cnt.sum()/np.maximum(1,cnt), 1.0).astype(np.float32)
    w[0]=0.0
    return torch.tensor(w, device=device)

bce = nn.BCEWithLogitsLoss()
ce  = nn.CrossEntropyLoss(reduction="sum")

def step(model, loader, train_df=None, opt=None):
    if train_df is not None:
        pos = int((train_df["y_gate"]==1).sum()); neg = int((train_df["y_gate"]==0).sum())
        bce.pos_weight = torch.tensor([(neg/max(1,pos))], device=device)
        ce.weight = action_weights(train_df)
    model.train(bool(opt))
    L=G=A=0.0; nB=0
    y_true=[]; y_pred=[]
    for x,yg,ya,mask in loader:
        g,a = model(x,mask)
        loss_g = bce(g[mask], yg[mask])
        pos_mask = (yg>0.5)&mask
        if pos_mask.any():
            loss_a = ce(a[pos_mask], ya[pos_mask]) / (pos_mask.sum().item())
        else:
            loss_a = torch.tensor(0.0, device=device)
        loss = loss_g + loss_a
        if opt:
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        L+=loss.item(); G+=loss_g.item(); A+=float(loss_a.item()); nB+=1
        if pos_mask.any():
            y_pred.extend(a[pos_mask].argmax(-1).detach().cpu().numpy().tolist())
            y_true.extend(ya[pos_mask].detach().cpu().numpy().tolist())
    acc = float(np.mean(np.array(y_pred)==np.array(y_true))) if y_true else float("nan")
    return {"loss":L/nB, "gate":G/nB, "act":A/nB, "act_only_acc":acc}

opt = torch.optim.AdamW(model_ga.parameters(), lr=1e-3)
for e in range(1,9):
    tr = step(model_ga, train_dl, train_df, opt)
    va = step(model_ga, val_dl,   train_df, None)
    print(f"[GA/7d] e{e}/8 | train L={tr['loss']:.3f} (G {tr['gate']:.3f}/A {tr['act']:.3f}) | val L={va['loss']:.3f} (G {va['gate']:.3f}/A {va['act']:.3f}) | act-only {va['act_only_acc']:.3f}")

# --- E) Calibrate gate threshold on validation (F1) ---
from sklearn.metrics import precision_recall_curve
@torch.no_grad()
def collect_scores(model, loader):
    s=[]; y=[]
    for x,yg,ya,mask in loader:
        g,_=model(x,mask); p=torch.sigmoid(g[mask]).cpu().numpy(); s.extend(p.tolist()); y.extend(yg[mask].cpu().numpy().tolist())
    return np.array(s), np.array(y)
scores, labels = collect_scores(model_ga, val_dl)
if labels.sum() in (0, len(labels)): tau = 0.5
else:
    ps,rs,ts = precision_recall_curve(labels, scores); f1=(2*ps*rs)/(ps+rs+1e-9); i=int(np.nanargmax(f1)); tau = float(ts[i]) if i < len(ts) else 0.5
print(f"[GA] calibrated gate τ={tau:.2f}")

# --- F) Rule-based ETA for demo (reads orders/results & returns next-result minutes) ---
def eta_rule(enc_id: str, now_min: int):
    # look up earliest result >= now for that encounter across all results of its day
    rows=[]
    # find the day from encounters
    day = int(enc_all.loc[enc_all["enc_id"]==enc_id, "day"].iloc[0])
    # get the matching tables for that day
    T = [t for t,_ in days][day]
    res = T["results"]; ords = T["orders"]; enc = T["encounters"]
    arr = pd.to_datetime(enc[enc["enc_id"]==enc_id]["arrival_ts"].iloc[0])
    now = arr + timedelta(minutes=int(now_min))
    fut = pd.to_datetime(res.loc[res["enc_id"]==enc_id, "ts_result"])
    fut = fut[fut>=now]
    if len(fut)==0: return float("nan")
    return (fut.min() - now).total_seconds()/60.0

ACTIONS = ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"]

@torch.no_grad()
def propose_ga(model, seq_df, k=4, gate_tau=0.5):
    seq = seq_df.sort_values("t")
    x = torch.from_numpy(seq[X_COLS].to_numpy(np.float32))[None,...].to(device)
    mask = torch.ones(1, x.shape[1], dtype=torch.bool, device=device)
    g,a = model(x,mask)
    gp = torch.sigmoid(g[0,-1]).item()
    probs = torch.softmax(a[0,-1], dim=-1).cpu().numpy()
    idx = np.argsort(probs)[::-1][:k]
    top = [(ACTIONS[i], float(probs[i])) for i in idx]
    # ETA via rule: use encounter id at this sequence and its current t
    enc_id = seq["enc_id"].iloc[0]; t_now = int(seq["t"].iloc[-1])
    eta_rb = eta_rule(enc_id, t_now)
    meta = {"gate_p": gp, "eta_first_result_min": float(eta_rb) if not math.isnan(eta_rb) else None}
    if gp < gate_tau: return [("none", 1.0)] + top, meta
    return top, meta

print("\n[GA] Ready. Use propose_ga(model_ga, seq_df, k=4, gate_tau=tau) on val/test sequences.")


Generated days: [286, 292, 270, 295, 297, 294, 288] total encounters: 2022
Feature cols added. Sample:
[GA/7d] e1/8 | train L=14.344 (G 1.061/A 13.284) | val L=12.977 (G 1.049/A 11.928) | act-only 0.554
[GA/7d] e2/8 | train L=12.326 (G 1.046/A 11.279) | val L=10.913 (G 1.025/A 9.887) | act-only 0.369
[GA/7d] e3/8 | train L=10.543 (G 1.025/A 9.518) | val L=9.687 (G 0.998/A 8.690) | act-only 0.357
[GA/7d] e4/8 | train L=9.630 (G 0.993/A 8.637) | val L=9.183 (G 0.982/A 8.201) | act-only 0.418
[GA/7d] e5/8 | train L=9.144 (G 0.961/A 8.184) | val L=8.616 (G 0.948/A 7.668) | act-only 0.530
[GA/7d] e6/8 | train L=8.890 (G 0.927/A 7.964) | val L=8.435 (G 0.883/A 7.552) | act-only 0.451
[GA/7d] e7/8 | train L=8.648 (G 0.867/A 7.781) | val L=8.226 (G 0.845/A 7.380) | act-only 0.548
[GA/7d] e8/8 | train L=8.456 (G 0.841/A 7.615) | val L=8.096 (G 0.797/A 7.299) | act-only 0.638
[GA] calibrated gate τ=0.58

[GA] Ready. Use propose_ga(model_ga, seq_df, k=4, gate_tau=tau) on val/test sequences.


In [121]:
import torch, json, pathlib
pathlib.Path("/mnt/data").mkdir(exist_ok=True)

torch.save(model_ga.state_dict(), "/mnt/data/ga_model.pt")
meta = {
    "arch": "GateActionNet(64)",
    "ACTIONS": ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"],
    "X_COLS": ["triage","complaint","hr","map","cap_icu","capacity_stale","t_norm"],
    "gate_tau": float(tau),
    "split": {"train_days":[0,1,2,3,4], "val_day":5, "test_day":6}
}
with open("/mnt/data/ga_meta.json","w") as f: json.dump(meta, f, indent=2)

print("Saved:", "/mnt/data/ga_model.pt", "/mnt/data/ga_meta.json")


Saved: /mnt/data/ga_model.pt /mnt/data/ga_meta.json


In [122]:
# Per-class report on first positive per encounter (val day)
import numpy as np
from sklearn.metrics import classification_report

ACTIONS = ["none","place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"]

labels, preds = [], []

for (d, enc), g in val_df.groupby(["day","enc_id"], sort=False):
    g = g.sort_values("t").reset_index(drop=True)
    idx = np.where(g["y_gate"].values == 1)[0]
    if len(idx) == 0: 
        continue
    i = idx[0]                          # first “act-now” moment for this case
    seq = g.iloc[:i+1]                  # what the model would have seen by then
    top, _ = propose_ga(model_ga, seq, k=1, gate_tau=0.0)   # force action choice for eval
    pred = ACTIONS.index(top[0][0]) if (top and top[0][0] in ACTIONS) else 0
    labels.append(int(g["y_action"].iloc[i]))
    preds.append(pred)

print("n positives:", len(labels))
print(classification_report(
    labels, preds, 
    labels=list(range(1, len(ACTIONS))),     # ignore 'none'
    target_names=ACTIONS[1:], 
    zero_division=0, digits=3
))


n positives: 294
                    precision    recall  f1-score   support

        place_labs      0.762     0.906     0.828       106
        place_xray      0.000     0.000     0.000        59
          place_ct      0.197     0.438     0.272        32
   page_cardiology      0.000     0.000     0.000         0
    page_neurology      0.000     0.000     0.000         0
prepare_admit_ward      1.000     1.000     1.000        97
 prepare_admit_icu      0.000     0.000     0.000         0

         micro avg      0.704     0.704     0.704       294
         macro avg      0.280     0.335     0.300       294
      weighted avg      0.626     0.704     0.658       294



In [123]:
# Single encounter demo on the held-out TEST day
import numpy as np

test_enc = test_df["enc_id"].unique()[0]
demo_seq = test_df[test_df.enc_id == test_enc].sort_values("t").reset_index(drop=True)

topk, meta = propose_ga(model_ga, demo_seq.iloc[:20], k=4, gate_tau=tau)
print("Top-k:", topk, "| meta:", meta)

# Choose an action with a floor to avoid junk (only orders/pages, not 'none')
action_floor = 0.15
action = "none"
for a, p in topk:
    if a in {"place_labs","place_xray","place_ct","page_cardiology","page_neurology","prepare_admit_ward","prepare_admit_icu"} and p >= action_floor:
        action = a; break
print("[DEMO] chosen:", action)

# Fire it through your bridge + notifier
try:
    orbis_sim.admit_patient("P-DEMO-GA")
    order_bridge("P-DEMO-GA", action, delay_sec=3)
    replay_loop(rn, cycles=6, sleep_sec=1)
except NameError:
    print("Bridge/notifier not loaded in this kernel; skip firing.")


Top-k: [('none', 1.0), ('prepare_admit_icu', 0.47244080901145935), ('prepare_admit_ward', 0.31797292828559875), ('page_cardiology', 0.1642673909664154), ('page_neurology', 0.041020940989255905)] | meta: {'gate_p': 0.1954549252986908, 'eta_first_result_min': None}
[DEMO] chosen: prepare_admit_icu
[BRIDGE] No ORM mapping for 'prepare_admit_icu'. Skipping.
[REPLAY] 1/6 | moved=0 oru=0 pushed=1
[REPLAY] 2/6 | moved=0 oru=0 pushed=0
[REPLAY] 3/6 | moved=0 oru=0 pushed=0
[REPLAY] 4/6 | moved=0 oru=0 pushed=0
[REPLAY] 5/6 | moved=0 oru=0 pushed=0
[REPLAY] 6/6 | moved=0 oru=0 pushed=0
